In [1]:
# Training 1 epoch

import os
import time
import torch
import torch_directml

# ==========================================
# 1. KONFIGURASI DEVICE DAN TRAINING
# ==========================================

device = torch_directml.device()

# Pastikan model berada di DirectML
model.to(device)
model.train()

# Optimizer SGD
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.001,
    momentum=0.9,
    weight_decay=0.0005
)

print("Device:", torch_directml.device_name(0))
print("Jumlah gambar training:", len(train_loader))
print("Batch size: 1")
print("Epoch: 1")
print("Learning rate: 0.001")

# ==========================================
# 2. FUNGSI UNTUK MENYIMPAN CHECKPOINT
# ==========================================

def move_to_cpu(data):
    if isinstance(data, torch.Tensor):
        return data.detach().cpu()
    elif isinstance(data, dict):
        return {k: move_to_cpu(v) for k, v in data.items()}
    elif isinstance(data, list):
        return [move_to_cpu(v) for v in data]
    elif isinstance(data, tuple):
        return tuple(move_to_cpu(v) for v in data)
    return data

# ==========================================
# 3. PROSES TRAINING 1 EPOCH
# ==========================================

total_loss = 0.0
start_time = time.time()

for batch_idx, (images, targets) in enumerate(train_loader):

    # Memindahkan gambar dan anotasi ke DirectML
    images = [img.to(device) for img in images]

    targets = [
        {
            key: value.to(device) if isinstance(value, torch.Tensor) else value
            for key, value in target.items()
        }
        for target in targets
    ]

    # Menghapus gradien sebelumnya
    optimizer.zero_grad()

    # Forward pass dan menghitung loss
    loss_dict = model(images, targets)
    losses = sum(loss for loss in loss_dict.values())

    # Memastikan loss valid
    if not torch.isfinite(losses).item():
        raise RuntimeError(
            f"Loss tidak valid pada batch {batch_idx + 1}: "
            f"{losses.item()}"
        )

    # Backward pass dan pembaruan bobot
    losses.backward()
    optimizer.step()

    batch_loss = losses.item()
    total_loss += batch_loss

    # Menampilkan progres setiap 100 batch
    if (batch_idx + 1) % 100 == 0 or (batch_idx + 1) == len(train_loader):
        print(
            f"Epoch 1 | Batch {batch_idx + 1}/{len(train_loader)} "
            f"| Loss: {batch_loss:.4f}"
        )

# ==========================================
# 4. HASIL TRAINING
# ==========================================

avg_loss = total_loss / len(train_loader)
elapsed_time = time.time() - start_time

print("\n========== HASIL TRAINING ==========")
print("Epoch              : 1")
print("Jumlah batch       :", len(train_loader))
print(f"Total loss         : {total_loss:.4f}")
print(f"Average loss       : {avg_loss:.4f}")
print(f"Durasi             : {elapsed_time / 60:.2f} menit")
print("====================================")

# ==========================================
# 5. SIMPAN CHECKPOINT
# ==========================================

checkpoint_dir = r"C:\Faster-RCNN-PPE\checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

checkpoint_path = os.path.join(
    checkpoint_dir,
    "fasterrcnn_epoch_1.pth"
)

checkpoint = {
    "epoch": 1,
    "model_state_dict": move_to_cpu(model.state_dict()),
    "optimizer_state_dict": move_to_cpu(optimizer.state_dict()),
    "average_loss": avg_loss,
    "learning_rate": 0.001
}

torch.save(checkpoint, checkpoint_path)

print("\nCheckpoint berhasil disimpan:")
print(checkpoint_path)

NameError: name 'model' is not defined

In [2]:
# Inisialisasi ulang model Faster R-CNN
import torch
import torch_directml
import torchvision

from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# 1. Menentukan device DirectML
device = torch_directml.device()

# 2. Memuat Faster R-CNN pretrained COCO
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT

model = fasterrcnn_resnet50_fpn(
    weights=weights,
    min_size=640,
    max_size=640
)

# 3. Mengganti classifier menjadi 9 kelas
# 0 = background
# 1-8 = kelas APD CHVG

in_features = model.roi_heads.box_predictor.cls_score.in_features

model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
    num_classes=9
)

# 4. Memindahkan model ke DirectML
model.to(device)

print("Model berhasil diinisialisasi!")
print("Device:", torch_directml.device_name(0))
print("Jumlah kelas:", model.roi_heads.box_predictor.cls_score.out_features)

Model berhasil diinisialisasi!
Device: AMD Radeon (TM) Graphics 
Jumlah kelas: 9


In [5]:
# Membuat Dataset class Faster R-CNN
from pathlib import Path
import xml.etree.ElementTree as ET

import torch
from torch.utils.data import Dataset
from PIL import Image
from torchvision.transforms.functional import to_tensor

# Lokasi dataset
ROOT = Path(r"C:\Faster-RCNN-PPE\dataset\images")

# Pemetaan kelas XML ke ID Faster R-CNN
CLASS_MAP = {
    "person": 1,
    "vest": 2,
    "yellow": 3,
    "white": 4,
    "glass": 5,
    "blue": 6,
    "red": 7,
    "head": 8
}

class CHVGPpeDataset(Dataset):

    def __init__(self, root, split):
        self.folder = Path(root) / split

        self.images = sorted([
            f for f in self.folder.iterdir()
            if f.is_file() and f.suffix.lower() in [".jpg", ".jpeg"]
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        image_path = self.images[idx]
        xml_path = self.folder / (image_path.stem + ".xml")

        # Baca gambar
        image = Image.open(image_path).convert("RGB")
        width, height = image.size

        # Baca XML
        root = ET.parse(xml_path).getroot()

        boxes = []
        labels = []

        for obj in root.findall("object"):

            class_name = obj.findtext("name")

            if class_name not in CLASS_MAP:
                raise ValueError(
                    f"Kelas tidak dikenal: {class_name}"
                )

            bbox = obj.find("bndbox")

            xmin = float(bbox.findtext("xmin"))
            ymin = float(bbox.findtext("ymin"))
            xmax = float(bbox.findtext("xmax"))
            ymax = float(bbox.findtext("ymax"))

            # Clipping koordinat ke batas gambar
            xmin = max(0, min(xmin, width))
            ymin = max(0, min(ymin, height))
            xmax = max(0, min(xmax, width))
            ymax = max(0, min(ymax, height))

            # Abaikan box yang menjadi tidak valid setelah clipping
            if xmin >= xmax or ymin >= ymax:
                continue

            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(CLASS_MAP[class_name])

        # Konversi gambar menjadi tensor
        image_tensor = to_tensor(image)

        # Tensor anotasi
        boxes_tensor = torch.tensor(
            boxes, dtype=torch.float32
        ).reshape(-1, 4)

        labels_tensor = torch.tensor(
            labels, dtype=torch.int64
        )

        target = {
            "boxes": boxes_tensor,
            "labels": labels_tensor,
            "image_id": torch.tensor([idx]),
            "area": (
                (boxes_tensor[:, 2] - boxes_tensor[:, 0]) *
                (boxes_tensor[:, 3] - boxes_tensor[:, 1])
            ),
            "iscrowd": torch.zeros(
                (len(labels),), dtype=torch.int64
            )
        }

        return image_tensor, target


# Membuat dataset untuk setiap split
train_dataset = CHVGPpeDataset(ROOT, "train")
val_dataset = CHVGPpeDataset(ROOT, "val")
test_dataset = CHVGPpeDataset(ROOT, "test")

print("Dataset berhasil dibuat")
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Dataset berhasil dibuat
Train: 1189
Validation: 170
Test: 340


In [6]:
# Membuat Data Loader
from torch.utils.data import DataLoader

def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

print("DataLoader berhasil dibuat")
print("Jumlah batch train:", len(train_loader))
print("Jumlah batch validation:", len(val_loader))

DataLoader berhasil dibuat
Jumlah batch train: 1189
Jumlah batch validation: 170


In [ ]:
# PERCOBAAN 1

In [7]:
# Training 1 epoch

import os
import time
import torch
import torch_directml

# ==========================================
# 1. KONFIGURASI DEVICE DAN TRAINING
# ==========================================

device = torch_directml.device()

# Pastikan model berada di DirectML
model.to(device)
model.train()

# Optimizer SGD
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.001,
    momentum=0.9,
    weight_decay=0.0005
)

print("Device:", torch_directml.device_name(0))
print("Jumlah gambar training:", len(train_loader))
print("Batch size: 1")
print("Epoch: 1")
print("Learning rate: 0.001")

# ==========================================
# 2. FUNGSI UNTUK MENYIMPAN CHECKPOINT
# ==========================================

def move_to_cpu(data):
    if isinstance(data, torch.Tensor):
        return data.detach().cpu()
    elif isinstance(data, dict):
        return {k: move_to_cpu(v) for k, v in data.items()}
    elif isinstance(data, list):
        return [move_to_cpu(v) for v in data]
    elif isinstance(data, tuple):
        return tuple(move_to_cpu(v) for v in data)
    return data

# ==========================================
# 3. PROSES TRAINING 1 EPOCH
# ==========================================

total_loss = 0.0
start_time = time.time()

for batch_idx, (images, targets) in enumerate(train_loader):

    # Memindahkan gambar dan anotasi ke DirectML
    images = [img.to(device) for img in images]

    targets = [
        {
            key: value.to(device) if isinstance(value, torch.Tensor) else value
            for key, value in target.items()
        }
        for target in targets
    ]

    # Menghapus gradien sebelumnya
    optimizer.zero_grad()

    # Forward pass dan menghitung loss
    loss_dict = model(images, targets)
    losses = sum(loss for loss in loss_dict.values())

    # Memastikan loss valid
    if not torch.isfinite(losses).item():
        raise RuntimeError(
            f"Loss tidak valid pada batch {batch_idx + 1}: "
            f"{losses.item()}"
        )

    # Backward pass dan pembaruan bobot
    losses.backward()
    optimizer.step()

    batch_loss = losses.item()
    total_loss += batch_loss

    # Menampilkan progres setiap 100 batch
    if (batch_idx + 1) % 100 == 0 or (batch_idx + 1) == len(train_loader):
        print(
            f"Epoch 1 | Batch {batch_idx + 1}/{len(train_loader)} "
            f"| Loss: {batch_loss:.4f}"
        )

# ==========================================
# 4. HASIL TRAINING
# ==========================================

avg_loss = total_loss / len(train_loader)
elapsed_time = time.time() - start_time

print("\n========== HASIL TRAINING ==========")
print("Epoch              : 1")
print("Jumlah batch       :", len(train_loader))
print(f"Total loss         : {total_loss:.4f}")
print(f"Average loss       : {avg_loss:.4f}")
print(f"Durasi             : {elapsed_time / 60:.2f} menit")
print("====================================")

# ==========================================
# 5. SIMPAN CHECKPOINT
# ==========================================

checkpoint_dir = r"C:\Faster-RCNN-PPE\checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

checkpoint_path = os.path.join(
    checkpoint_dir,
    "fasterrcnn_epoch_1.pth"
)

checkpoint = {
    "epoch": 1,
    "model_state_dict": move_to_cpu(model.state_dict()),
    "optimizer_state_dict": move_to_cpu(optimizer.state_dict()),
    "average_loss": avg_loss,
    "learning_rate": 0.001
}

torch.save(checkpoint, checkpoint_path)

print("\nCheckpoint berhasil disimpan:")
print(checkpoint_path)

Device: AMD Radeon (TM) Graphics 
Jumlah gambar training: 1189
Batch size: 1
Epoch: 1
Learning rate: 0.001


C:\Users\ANGELINA\miniconda3\envs\pytorch-directml\lib\site-packages\torch\nn\functional.py:3244: UserWarning: The operator 'aten::log_sigmoid_forward' is not currently supported on the DML backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at C:\__w\1\s\pytorch-directml-plugin\torch_directml\csrc\dml\dml_cpu_fallback.cpp:17.)
  return torch.binary_cross_entropy_with_logits(input, target, weight, pos_weight, reduction_enum)


Epoch 1 | Batch 100/1189 | Loss: 0.5146
Epoch 1 | Batch 200/1189 | Loss: 0.8206
Epoch 1 | Batch 300/1189 | Loss: 0.8590
Epoch 1 | Batch 400/1189 | Loss: 0.3616
Epoch 1 | Batch 500/1189 | Loss: 0.6547
Epoch 1 | Batch 600/1189 | Loss: 0.5946


IndexError: index 0 is out of bounds for dimension 0 with size 0

In [8]:
# Periksa seluruh bounding box training
invalid_images = []

for idx in range(len(train_dataset)):

    image, target = train_dataset[idx]

    boxes = target["boxes"]

    # Pemeriksaan bounding box pada CPU
    if len(boxes) > 0:

        invalid = (
            (boxes[:, 2] <= boxes[:, 0]) |
            (boxes[:, 3] <= boxes[:, 1])
        )

        if invalid.any():
            invalid_images.append({
                "index": idx,
                "boxes": boxes[invalid].tolist()
            })

print("===== HASIL PEMERIKSAAN BOUNDING BOX =====")
print("Jumlah gambar training:", len(train_dataset))
print("Jumlah gambar dengan bounding box tidak valid:",
      len(invalid_images))

if invalid_images:
    print("\nContoh bounding box bermasalah:")
    print(invalid_images[:5])
else:
    print("\nSemua bounding box training valid.")

===== HASIL PEMERIKSAAN BOUNDING BOX =====
Jumlah gambar training: 1189
Jumlah gambar dengan bounding box tidak valid: 0

Semua bounding box training valid.


In [9]:
# Kompabilitas operasi DirectML
import torch
import torch_directml

device = torch_directml.device()

# Membuat bounding box valid
boxes = torch.tensor(
    [
        [10, 20, 100, 150],
        [30, 40, 120, 180],
        [50, 60, 200, 250]
    ],
    dtype=torch.float32
).to(device)

# Pemeriksaan seperti internal Faster R-CNN
degenerate_boxes = boxes[:, 2:] <= boxes[:, :2]

print("Device:", boxes.device)
print("\nDegenerate boxes:")
print(degenerate_boxes)

print("\nHasil any:")
print(degenerate_boxes.any(dim=1))

print("\nHasil where:")
print(torch.where(degenerate_boxes.any(dim=1))[0])

Device: privateuseone:0

Degenerate boxes:
tensor([[False, False],
        [False, False],
        [False, False]], device='privateuseone:0')

Hasil any:
tensor([False, False, False], device='privateuseone:0')

Hasil where:
tensor([], device='privateuseone:0', dtype=torch.int64)


In [10]:
# Periksa batch training yang bermasalah
import torch

print("===== PEMERIKSAAN TARGET TRAINING =====")

for batch_idx, (images, targets) in enumerate(train_loader):

    for target_idx, target in enumerate(targets):

        boxes = target["boxes"]

        if len(boxes) == 0:
            continue

        # Pemeriksaan di CPU
        invalid = (
            (boxes[:, 2] <= boxes[:, 0]) |
            (boxes[:, 3] <= boxes[:, 1])
        )

        if invalid.any():
            print("Bounding box tidak valid ditemukan!")
            print("Batch:", batch_idx + 1)
            print("Target:", target_idx)
            print(boxes[invalid])
            raise ValueError("Bounding box tidak valid")

print("Semua target training berhasil diperiksa.")

===== PEMERIKSAAN TARGET TRAINING =====
Semua target training berhasil diperiksa.


In [11]:
# Pemeriksaan internal torchvision
print("Torch version:", torch.__version__)

import torchvision
print("Torchvision version:", torchvision.__version__)

print("Device:", device)
print("Model training mode:", model.training)
print("Jumlah kelas:", model.roi_heads.box_predictor.cls_score.out_features)

Torch version: 2.4.1+cpu
Torchvision version: 0.19.1+cpu
Device: privateuseone:0
Model training mode: True
Jumlah kelas: 9


In [12]:
# Pengujian forward pass satu batch
# Ambil satu batch dari train_loader
images, targets = next(iter(train_loader))

# Pindahkan ke DirectML
images = [img.to(device) for img in images]

targets = [
    {
        key: value.to(device) if isinstance(value, torch.Tensor) else value
        for key, value in target.items()
    }
    for target in targets
]

print("Jumlah gambar:", len(images))
print("Jumlah bounding box:", len(targets[0]["boxes"]))

# Periksa bounding box pada device DirectML
boxes = targets[0]["boxes"]

degenerate_boxes = boxes[:, 2:] <= boxes[:, :2]

print("Hasil pemeriksaan:")
print(degenerate_boxes)

print("Hasil any:")
print(degenerate_boxes.any(dim=1))

print("Hasil where:")
print(torch.where(degenerate_boxes.any(dim=1))[0])

# Uji forward pass
model.train()

with torch.no_grad():
    loss_dict = model(images, targets)

print("\nForward pass berhasil!")
print(loss_dict)

Jumlah gambar: 1
Jumlah bounding box: 7
Hasil pemeriksaan:
tensor([[False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False]], device='privateuseone:0')
Hasil any:
tensor([False, False, False, False, False, False, False],
       device='privateuseone:0')
Hasil where:
tensor([], device='privateuseone:0', dtype=torch.int64)

Forward pass berhasil!
{'loss_classifier': tensor(0.2838, device='privateuseone:0'), 'loss_box_reg': tensor(0.4813, device='privateuseone:0'), 'loss_objectness': tensor(0.0173, device='privateuseone:0'), 'loss_rpn_box_reg': tensor(0.0267, device='privateuseone:0')}


In [13]:
# Periksa model dan optimizer
print("Model tersedia:", "model" in globals())
print("Optimizer tersedia:", "optimizer" in globals())

if "model" in globals():
    print("Mode model:", model.training)

if "optimizer" in globals():
    print("Jenis optimizer:", type(optimizer).__name__)

Model tersedia: True
Optimizer tersedia: True
Mode model: True
Jenis optimizer: SGD


In [ ]:
# PERCOBAAN 2

In [14]:
# Muat ulang model petrained COCO
import torch
import torch_directml
import torchvision

from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# 1. Menentukan device DirectML
device = torch_directml.device()

# 2. Memuat model pretrained COCO
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT

model = fasterrcnn_resnet50_fpn(
    weights=weights,
    min_size=640,
    max_size=640
)

# 3. Mengganti classifier menjadi 9 kelas
in_features = model.roi_heads.box_predictor.cls_score.in_features

model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
    num_classes=9
)

# 4. Memindahkan model ke DirectML
model.to(device)

print("Model berhasil dimuat ulang!")
print("Device:", torch_directml.device_name(0))
print("Jumlah kelas:",
      model.roi_heads.box_predictor.cls_score.out_features)
print("Model training mode:", model.training)

Model berhasil dimuat ulang!
Device: AMD Radeon (TM) Graphics 
Jumlah kelas: 9
Model training mode: True


In [15]:
# Membuat optimizer baru
import torch

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.001,
    momentum=0.9,
    weight_decay=0.0005
)

print("Optimizer:", type(optimizer).__name__)
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Momentum:", optimizer.param_groups[0]["momentum"])
print("Weight decay:", optimizer.param_groups[0]["weight_decay"])
print("Model training mode:", model.training)

Optimizer: SGD
Learning rate: 0.001
Momentum: 0.9
Weight decay: 0.0005
Model training mode: True


In [16]:
# Cek dataloader
print("Train dataset tersedia:", "train_dataset" in globals())
print("Train loader tersedia:", "train_loader" in globals())

print("Jumlah gambar training:", len(train_dataset))
print("Jumlah batch training:", len(train_loader))
print("Batch size:", train_loader.batch_size)

Train dataset tersedia: True
Train loader tersedia: True
Jumlah gambar training: 1189
Jumlah batch training: 1189
Batch size: 1


In [17]:
# Uji 1 batch
images, targets = next(iter(train_loader))

images = [img.to(device) for img in images]

targets = [
    {
        key: value.to(device) if isinstance(value, torch.Tensor) else value
        for key, value in target.items()
    }
    for target in targets
]

model.train()

loss_dict = model(images, targets)

print("Forward pass berhasil!")
print("Jumlah gambar:", len(images))
print("Jumlah bounding box:", len(targets[0]["boxes"]))
print("\nRincian loss:")

for name, value in loss_dict.items():
    print(f"{name}: {value.item():.6f}")

total_loss = sum(loss for loss in loss_dict.values())
print("\nTotal loss:", total_loss.item())

Forward pass berhasil!
Jumlah gambar: 1
Jumlah bounding box: 19

Rincian loss:
loss_classifier: 2.207328
loss_box_reg: 0.847479
loss_objectness: 0.295899
loss_rpn_box_reg: 0.108495

Total loss: 3.459199905395508


In [18]:
# Uji backward & optimizer
model.train()

optimizer.zero_grad()

loss_dict = model(images, targets)
total_loss = sum(loss for loss in loss_dict.values())

print("Total loss sebelum backward:", total_loss.item())

total_loss.backward()

print("Backward berhasil!")

optimizer.step()

print("Optimizer step berhasil!")
print("Pengujian satu langkah training selesai.")

Total loss sebelum backward: 3.5085740089416504
Backward berhasil!
Optimizer step berhasil!
Pengujian satu langkah training selesai.


In [19]:
# Training 1 epoch
import time
import torch
from pathlib import Path

model.train()

num_epochs = 1

for epoch in range(num_epochs):
    start_time = time.time()
    total_loss_epoch = 0.0

    for batch_idx, (images, targets) in enumerate(train_loader):

        images = [img.to(device) for img in images]

        targets = [
            {
                key: value.to(device) if isinstance(value, torch.Tensor) else value
                for key, value in target.items()
            }
            for target in targets
        ]

        optimizer.zero_grad()

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        losses.backward()
        optimizer.step()

        total_loss_epoch += losses.item()

        if (batch_idx + 1) % 100 == 0:
            print(
                f"Epoch {epoch+1} | "
                f"Batch {batch_idx+1}/{len(train_loader)} | "
                f"Loss: {losses.item():.4f}"
            )

    avg_loss = total_loss_epoch / len(train_loader)
    elapsed_time = time.time() - start_time

    print("\n===== HASIL TRAINING =====")
    print("Epoch:", epoch + 1)
    print("Average Loss:", avg_loss)
    print("Waktu training:", round(elapsed_time / 60, 2), "menit")

Epoch 1 | Batch 100/1189 | Loss: 1.6179
Epoch 1 | Batch 200/1189 | Loss: 0.9289


RuntimeError: output with shape [] doesn't match the broadcast shape [0, 4]

In [20]:
# Periksa batch yang menyebabkan eror
import torch
import torch.nn.functional as F

print("===== BATCH YANG GAGAL =====")
print("Nomor batch:", batch_idx + 1)
print("Jumlah gambar:", len(images))
print("Jumlah bounding box:", len(targets[0]["boxes"]))
print("Device gambar:", images[0].device)
print("Device bounding box:", targets[0]["boxes"].device)

print("\n===== UJI OPERASI SMOOTH L1 DIRECTML =====")

try:
    pred = torch.empty((0, 4), device=device)
    target = torch.empty((0, 4), device=device)

    result = F.smooth_l1_loss(
        pred,
        target,
        beta=1/9,
        reduction="sum"
    )

    print("Smooth L1 berhasil:", result.item())

except Exception as e:
    print("Smooth L1 gagal:")
    print(type(e).__name__, str(e))

===== BATCH YANG GAGAL =====
Nomor batch: 256
Jumlah gambar: 1
Jumlah bounding box: 0
Device gambar: privateuseone:0
Device bounding box: privateuseone:0

===== UJI OPERASI SMOOTH L1 DIRECTML =====
Smooth L1 gagal:
RuntimeError output with shape [] doesn't match the broadcast shape [0, 4]


In [21]:
# Eror training disebabkan adanya gambar yang tidak memiliki label objek apd
# Identifikasi gambar yang tidak ada
image_index = targets[0]["image_id"].item()

image_path = train_dataset.images[image_index]
xml_path = image_path.with_suffix(".xml")

print("Indeks dataset:", image_index)
print("Nama gambar:", image_path.name)
print("Lokasi XML:", xml_path)
print("Jumlah bounding box:", len(targets[0]["boxes"]))

Indeks dataset: 381
Nama gambar: ppe_0169_jpg.rf.d85b8729987cec65187b118f9e8c6328.jpg
Lokasi XML: C:\Faster-RCNN-PPE\dataset\images\train\ppe_0169_jpg.rf.d85b8729987cec65187b118f9e8c6328.xml
Jumlah bounding box: 0


In [22]:
# Pasang path pada RPN
import torch
import torch.nn.functional as F
from types import MethodType

def compute_loss_safe(self, objectness, pred_bbox_deltas, labels, regression_targets):

    # Memilih anchor positif dan negatif
    sampled_pos_inds, sampled_neg_inds = self.fg_bg_sampler(labels)

    sampled_pos_inds = torch.where(
        torch.cat(sampled_pos_inds, dim=0)
    )[0]

    sampled_neg_inds = torch.where(
        torch.cat(sampled_neg_inds, dim=0)
    )[0]

    sampled_inds = torch.cat([
        sampled_pos_inds,
        sampled_neg_inds
    ])

    objectness = objectness.flatten()
    labels = torch.cat(labels, dim=0)
    regression_targets = torch.cat(regression_targets, dim=0)

    # Penanganan khusus jika tidak ada anchor positif
    if sampled_pos_inds.numel() == 0:
        box_loss = pred_bbox_deltas.sum() * 0.0
    else:
        box_loss = F.smooth_l1_loss(
            pred_bbox_deltas[sampled_pos_inds],
            regression_targets[sampled_pos_inds],
            beta=1 / 9,
            reduction="sum"
        )

    # Normalisasi box loss
    box_loss = box_loss / max(sampled_inds.numel(), 1)

    # Objectness loss tetap dihitung
    if sampled_inds.numel() == 0:
        objectness_loss = objectness.sum() * 0.0
    else:
        objectness_loss = F.binary_cross_entropy_with_logits(
            objectness[sampled_inds],
            labels[sampled_inds]
        )

    return objectness_loss, box_loss


# Memasang patch pada RPN model saat ini
model.rpn.compute_loss = MethodType(
    compute_loss_safe,
    model.rpn
)

print("Patch RPN berhasil dipasang.")
print("Faster R-CNN tetap menggunakan arsitektur yang sama.")

Patch RPN berhasil dipasang.
Faster R-CNN tetap menggunakan arsitektur yang sama.


In [23]:
# Uji forward pass pada gambar bermasalah
# Mengambil gambar yang sebelumnya menyebabkan error
image_index = 381

image, target = train_dataset[image_index]

images_test = [image.to(device)]

targets_test = [
    {
        key: value.to(device) if isinstance(value, torch.Tensor) else value
        for key, value in target.items()
    }
]

print("Nama gambar:", train_dataset.images[image_index].name)
print("Jumlah bounding box:", len(target["boxes"]))

model.train()

try:
    loss_dict = model(images_test, targets_test)

    total_loss = sum(loss for loss in loss_dict.values())

    print("\nForward pass berhasil!")
    print("Rincian loss:")

    for name, value in loss_dict.items():
        print(f"{name}: {value.item():.6f}")

    print("Total loss:", total_loss.item())

except Exception as e:
    print("\nForward pass gagal!")
    print(type(e).__name__, str(e))

Nama gambar: ppe_0169_jpg.rf.d85b8729987cec65187b118f9e8c6328.jpg
Jumlah bounding box: 0

Forward pass gagal!
IndexError index 0 is out of bounds for dimension 0 with size 0


In [24]:
import traceback

try:
    model.train()
    loss_dict = model(images_test, targets_test)

    print("Forward pass berhasil!")

except Exception:
    print("===== TRACEBACK LENGKAP =====")
    traceback.print_exc()

===== TRACEBACK LENGKAP =====


Traceback (most recent call last):
  File "C:\Users\ANGELINA\AppData\Local\Temp\ipykernel_20712\193026700.py", line 5, in <module>
    loss_dict = model(images_test, targets_test)
  File "C:\Users\ANGELINA\miniconda3\envs\pytorch-directml\lib\site-packages\torch\nn\modules\module.py", line 1553, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "C:\Users\ANGELINA\miniconda3\envs\pytorch-directml\lib\site-packages\torch\nn\modules\module.py", line 1562, in _call_impl
    return forward_call(*args, **kwargs)
  File "C:\Users\ANGELINA\miniconda3\envs\pytorch-directml\lib\site-packages\torchvision\models\detection\generalized_rcnn.py", line 93, in forward
    bb_idx = torch.where(degenerate_boxes.any(dim=1))[0][0]
IndexError: index 0 is out of bounds for dimension 0 with size 0


In [25]:
# Atur mempertahankan pemeriksaan bounding box tidak valid untuk gambar yang memiliki objek. Untuk gambar tanpa objek, pemeriksaan dilewati.
from collections import OrderedDict
from types import MethodType
import torch

def forward_safe(self, images, targets=None):

    if self.training and targets is None:
        raise ValueError("Targets harus diberikan saat training.")

    if self.training:
        for target in targets:
            boxes = target["boxes"]

            if isinstance(boxes, torch.Tensor) and boxes.numel() > 0:

                degenerate_boxes = boxes[:, 2:] <= boxes[:, :2]

                if degenerate_boxes.any():
                    bb_idx = torch.where(
                        degenerate_boxes.any(dim=1)
                    )[0][0]

                    raise ValueError(
                        f"Bounding box tidak valid pada indeks {bb_idx}"
                    )

    original_image_sizes = []

    for img in images:
        val = img.shape[-2:]
        original_image_sizes.append((val[0], val[1]))

    images, targets = self.transform(images, targets)

    features = self.backbone(images.tensors)

    if isinstance(features, torch.Tensor):
        features = OrderedDict([("0", features)])

    proposals, proposal_losses = self.rpn(
        images, features, targets
    )

    detections, detector_losses = self.roi_heads(
        features,
        proposals,
        images.image_sizes,
        targets
    )

    detections = self.transform.postprocess(
        detections,
        images.image_sizes,
        original_image_sizes
    )

    losses = {}
    losses.update(detector_losses)
    losses.update(proposal_losses)

    return self.eager_outputs(losses, detections)


model.forward = MethodType(forward_safe, model)

print("Patch pemeriksaan bounding box berhasil dipasang.")

Patch pemeriksaan bounding box berhasil dipasang.


In [26]:
# Uji ulang gambar bermasalah
model.train()

try:
    loss_dict = model(images_test, targets_test)

    total_loss = sum(loss for loss in loss_dict.values())

    print("Forward pass berhasil!")
    print("\nRincian loss:")

    for name, value in loss_dict.items():
        print(f"{name}: {value.item():.6f}")

    print("\nTotal loss:", total_loss.item())

except Exception:
    import traceback
    print("Forward pass gagal!")
    traceback.print_exc()

Forward pass gagal!


Traceback (most recent call last):
  File "C:\Users\ANGELINA\AppData\Local\Temp\ipykernel_20712\1356883059.py", line 5, in <module>
    loss_dict = model(images_test, targets_test)
  File "C:\Users\ANGELINA\miniconda3\envs\pytorch-directml\lib\site-packages\torch\nn\modules\module.py", line 1553, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "C:\Users\ANGELINA\miniconda3\envs\pytorch-directml\lib\site-packages\torch\nn\modules\module.py", line 1562, in _call_impl
    return forward_call(*args, **kwargs)
  File "C:\Users\ANGELINA\AppData\Local\Temp\ipykernel_20712\2872708633.py", line 45, in forward_safe
    detections, detector_losses = self.roi_heads(
  File "C:\Users\ANGELINA\miniconda3\envs\pytorch-directml\lib\site-packages\torch\nn\modules\module.py", line 1553, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "C:\Users\ANGELINA\miniconda3\envs\pytorch-directml\lib\site-packages\torch\nn\modules\module.py", line 1562, in _call_imp

In [27]:
# menyesuaikan fungsi add_gt_proposals() agar tidak menjalankan torch.cat() ketika ground truth kosong
from types import MethodType

def add_gt_proposals_safe(self, proposals, targets):

    gt_boxes = [target["boxes"] for target in targets]

    proposals = [
        proposal if gt_box.numel() == 0
        else torch.cat((proposal, gt_box))
        for proposal, gt_box in zip(proposals, gt_boxes)
    ]

    return proposals


model.roi_heads.add_gt_proposals = MethodType(
    add_gt_proposals_safe,
    model.roi_heads
)

print("Patch ROI Heads berhasil dipasang.")

Patch ROI Heads berhasil dipasang.


In [28]:
# Uji ulang kesekian gambar problematik
model.train()

try:
    loss_dict = model(images_test, targets_test)

    total_loss = sum(loss for loss in loss_dict.values())

    print("Forward pass berhasil!")
    print("\nRincian loss:")

    for name, value in loss_dict.items():
        print(f"{name}: {value.item():.6f}")

    print("\nTotal loss:", total_loss.item())

except Exception:
    import traceback
    print("Forward pass gagal!")
    traceback.print_exc()

Forward pass gagal!


Traceback (most recent call last):
  File "C:\Users\ANGELINA\AppData\Local\Temp\ipykernel_20712\1163614037.py", line 5, in <module>
    loss_dict = model(images_test, targets_test)
  File "C:\Users\ANGELINA\miniconda3\envs\pytorch-directml\lib\site-packages\torch\nn\modules\module.py", line 1553, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "C:\Users\ANGELINA\miniconda3\envs\pytorch-directml\lib\site-packages\torch\nn\modules\module.py", line 1562, in _call_impl
    return forward_call(*args, **kwargs)
  File "C:\Users\ANGELINA\AppData\Local\Temp\ipykernel_20712\2872708633.py", line 45, in forward_safe
    detections, detector_losses = self.roi_heads(
  File "C:\Users\ANGELINA\miniconda3\envs\pytorch-directml\lib\site-packages\torch\nn\modules\module.py", line 1553, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "C:\Users\ANGELINA\miniconda3\envs\pytorch-directml\lib\site-packages\torch\nn\modules\module.py", line 1562, in _call_imp

In [29]:
# Perbaiki patch ROI Heads
from types import MethodType
import torch

def add_gt_proposals_safe(self, proposals, gt_boxes):

    updated_proposals = []

    for proposal, gt_box in zip(proposals, gt_boxes):

        if gt_box.numel() == 0:
            updated_proposals.append(proposal)
        else:
            updated_proposals.append(
                torch.cat((proposal, gt_box), dim=0)
            )

    return updated_proposals


model.roi_heads.add_gt_proposals = MethodType(
    add_gt_proposals_safe,
    model.roi_heads
)

print("Patch ROI Heads berhasil diperbaiki.")

Patch ROI Heads berhasil diperbaiki.


In [30]:
# Uji lagi harus sukses
model.train()

try:
    loss_dict = model(images_test, targets_test)

    total_loss = sum(loss for loss in loss_dict.values())

    print("Forward pass berhasil!")
    print("\nRincian loss:")

    for name, value in loss_dict.items():
        print(f"{name}: {value.item():.6f}")

    print("\nTotal loss:", total_loss.item())

except Exception:
    import traceback
    print("Forward pass gagal!")
    traceback.print_exc()

Forward pass gagal!


Traceback (most recent call last):
  File "C:\Users\ANGELINA\AppData\Local\Temp\ipykernel_20712\2265902306.py", line 5, in <module>
    loss_dict = model(images_test, targets_test)
  File "C:\Users\ANGELINA\miniconda3\envs\pytorch-directml\lib\site-packages\torch\nn\modules\module.py", line 1553, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "C:\Users\ANGELINA\miniconda3\envs\pytorch-directml\lib\site-packages\torch\nn\modules\module.py", line 1562, in _call_impl
    return forward_call(*args, **kwargs)
  File "C:\Users\ANGELINA\AppData\Local\Temp\ipykernel_20712\2872708633.py", line 45, in forward_safe
    detections, detector_losses = self.roi_heads(
  File "C:\Users\ANGELINA\miniconda3\envs\pytorch-directml\lib\site-packages\torch\nn\modules\module.py", line 1553, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "C:\Users\ANGELINA\miniconda3\envs\pytorch-directml\lib\site-packages\torch\nn\modules\module.py", line 1562, in _call_imp

In [31]:
# Perbaiki Fast R-CNN loss
import torch
import torch.nn.functional as F
import torchvision.models.detection.roi_heads as roi_heads_module

def fastrcnn_loss_safe(
    class_logits,
    box_regression,
    labels,
    regression_targets
):
    labels = torch.cat(labels, dim=0)
    regression_targets = torch.cat(regression_targets, dim=0)

    # Classification loss tetap dihitung
    classification_loss = F.cross_entropy(
        class_logits,
        labels
    )

    # Memilih proposal positif (kelas > 0)
    sampled_pos_inds = torch.where(labels > 0)[0]

    labels_pos = labels[sampled_pos_inds]

    N, num_classes = class_logits.shape

    box_regression = box_regression.reshape(
        N, box_regression.size(-1) // 4, 4
    )

    # Penanganan jika tidak ada proposal positif
    if sampled_pos_inds.numel() == 0:
        box_loss = box_regression.sum() * 0.0

    else:
        box_loss = F.smooth_l1_loss(
            box_regression[sampled_pos_inds, labels_pos],
            regression_targets[sampled_pos_inds],
            beta=1 / 9,
            reduction="sum"
        )

    box_loss = box_loss / max(labels.numel(), 1)

    return classification_loss, box_loss


# Ganti fungsi Fast R-CNN loss yang digunakan Torchvision
roi_heads_module.fastrcnn_loss = fastrcnn_loss_safe

print("Patch Fast R-CNN loss berhasil dipasang.")

Patch Fast R-CNN loss berhasil dipasang.


In [32]:
# Uji
model.train()

with torch.no_grad():
    loss_dict = model(images_test, targets_test)

total_loss = sum(loss for loss in loss_dict.values())

print("Forward pada gambar tanpa bounding box berhasil.")
print("Loss per komponen:")

for name, loss in loss_dict.items():
    print(f"{name}: {loss.item():.6f}")

print(f"Total loss: {total_loss.item():.6f}")

Forward pada gambar tanpa bounding box berhasil.
Loss per komponen:
loss_classifier: 0.045018
loss_box_reg: 0.000000
loss_objectness: 0.000195
loss_rpn_box_reg: 0.000000
Total loss: 0.045213


In [33]:
# Uji backward & optimizer gambar problematik
model.train()
optimizer.zero_grad()

loss_dict = model(images_test, targets_test)
total_loss = sum(loss for loss in loss_dict.values())

print(f"Total loss sebelum backward: {total_loss.item():.6f}")

total_loss.backward()
print("Backward berhasil.")

optimizer.step()
print("Optimizer step berhasil.")

Total loss sebelum backward: 0.026851
Backward berhasil.
Optimizer step berhasil.


In [34]:
# Uji gambar yang memiliki bounding box
sample_index = 0

image_test_positive, target_test_positive = train_dataset[sample_index]

print("Nama gambar:", train_dataset.images[sample_index])
print("Jumlah bounding box:", len(target_test_positive["boxes"]))

images_positive = [image_test_positive.to(device)]

targets_positive = [{
    key: value.to(device) if isinstance(value, torch.Tensor) else value
    for key, value in target_test_positive.items()
}]

model.train()

with torch.no_grad():
    loss_dict_positive = model(images_positive, targets_positive)

print("\nLoss pada gambar dengan objek:")

for name, loss in loss_dict_positive.items():
    print(f"{name}: {loss.item():.6f}")

print("Total loss:", sum(loss_dict_positive.values()).item())

Nama gambar: C:\Faster-RCNN-PPE\dataset\images\train\000004_jpg.rf.a64a312cbd6f217ceb7680f99531fcd6.jpg
Jumlah bounding box: 11

Loss pada gambar dengan objek:
loss_classifier: 0.408500
loss_box_reg: 0.695038
loss_objectness: 0.010193
loss_rpn_box_reg: 0.089168
Total loss: 1.2028982639312744


In [35]:
# Uji backward
model.train()
optimizer.zero_grad()

loss_dict_positive = model(images_positive, targets_positive)
total_loss_positive = sum(loss_dict_positive.values())

print(f"Total loss sebelum backward: {total_loss_positive.item():.6f}")

total_loss_positive.backward()
print("Backward berhasil.")

optimizer.step()
print("Optimizer step berhasil.")

Total loss sebelum backward: 1.198434
Backward berhasil.
Optimizer step berhasil.


In [ ]:
# PERSIAPAN TRAINING

In [36]:
# cek variabel
variabel_yang_dicek = [
    "train_dataset",
    "val_dataset",
    "test_dataset",
    "train_loader",
    "val_loader",
    "test_loader",
    "model",
    "optimizer",
    "device"
]

for nama in variabel_yang_dicek:
    tersedia = nama in globals()
    print(f"{nama:15} : {'Tersedia' if tersedia else 'Tidak tersedia'}")

train_dataset   : Tersedia
val_dataset     : Tersedia
test_dataset    : Tersedia
train_loader    : Tersedia
val_loader      : Tersedia
test_loader     : Tidak tersedia
model           : Tersedia
optimizer       : Tersedia
device          : Tersedia


In [37]:
# cek konfigurasi dataloader
print("=== KONFIGURASI DATASET DAN DATALOADER ===")

print("Jumlah data training :", len(train_dataset))
print("Jumlah data validasi :", len(val_dataset))
print("Jumlah data testing  :", len(test_dataset))

print("\nJumlah batch training :", len(train_loader))
print("Jumlah batch validasi :", len(val_loader))

print("\nBatch size training :", train_loader.batch_size)
print("Batch size validasi :", val_loader.batch_size)

print("\nJumlah worker training :", train_loader.num_workers)
print("Jumlah worker validasi :", val_loader.num_workers)

=== KONFIGURASI DATASET DAN DATALOADER ===
Jumlah data training : 1189
Jumlah data validasi : 170
Jumlah data testing  : 340

Jumlah batch training : 1189
Jumlah batch validasi : 170

Batch size training : 1
Batch size validasi : 1

Jumlah worker training : 0
Jumlah worker validasi : 0


In [38]:
# Inisialisasi
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# Pastikan perangkat DirectML tetap digunakan
print("Device:", device)

# Muat model pretrained COCO
weights = torchvision.models.detection.FasterRCNN_ResNet50_FPN_Weights.DEFAULT

model = fasterrcnn_resnet50_fpn(
    weights=weights,
    min_size=640,
    max_size=640
)

# Ganti predictor untuk 8 kelas APD + background
in_features = model.roi_heads.box_predictor.cls_score.in_features

model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
    num_classes=9
)

# Pindahkan model ke DirectML
model.to(device)

# Buat optimizer baru
optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=0.001,
    momentum=0.9,
    weight_decay=0.0005
)

print("\nModel berhasil diinisialisasi ulang.")
print("Jumlah kelas:", model.roi_heads.box_predictor.cls_score.out_features)
print("Device model:", next(model.parameters()).device)
print("Optimizer:", type(optimizer).__name__)

Device: privateuseone:0

Model berhasil diinisialisasi ulang.
Jumlah kelas: 9
Device model: privateuseone:0
Optimizer: SGD


In [39]:
# Pasang patch ke DirectML
from types import MethodType
from collections import OrderedDict

import torch
import torch.nn.functional as F
import torchvision.models.detection.roi_heads as roi_heads_module


# ==================================================
# 1. PATCH RPN LOSS
# Menangani kondisi tidak ada anchor positif
# ==================================================

def compute_loss_safe(self, objectness, pred_bbox_deltas, labels, regression_targets):

    sampled_pos_inds, sampled_neg_inds = self.fg_bg_sampler(labels)

    sampled_pos_inds = torch.where(torch.cat(sampled_pos_inds, dim=0))[0]
    sampled_neg_inds = torch.where(torch.cat(sampled_neg_inds, dim=0))[0]

    sampled_inds = torch.cat([sampled_pos_inds, sampled_neg_inds])

    objectness = objectness.flatten()
    labels = torch.cat(labels, dim=0)
    regression_targets = torch.cat(regression_targets, dim=0)

    if sampled_pos_inds.numel() == 0:
        box_loss = pred_bbox_deltas.sum() * 0.0
    else:
        box_loss = F.smooth_l1_loss(
            pred_bbox_deltas[sampled_pos_inds],
            regression_targets[sampled_pos_inds],
            beta=1 / 9,
            reduction="sum"
        )

    box_loss = box_loss / max(sampled_inds.numel(), 1)

    if sampled_inds.numel() == 0:
        objectness_loss = objectness.sum() * 0.0
    else:
        objectness_loss = F.binary_cross_entropy_with_logits(
            objectness[sampled_inds],
            labels[sampled_inds]
        )

    return objectness_loss, box_loss


model.rpn.compute_loss = MethodType(compute_loss_safe, model.rpn)


# ==================================================
# 2. PATCH GENERALIZED RCNN FORWARD
# Menghindari error pada pemeriksaan bounding box kosong
# ==================================================

def forward_safe(self, images, targets=None):

    if self.training and targets is None:
        raise ValueError("Targets harus diberikan saat training.")

    if self.training:
        for target in targets:
            boxes = target["boxes"]

            if isinstance(boxes, torch.Tensor) and boxes.numel() > 0:
                degenerate_boxes = boxes[:, 2:] <= boxes[:, :2]

                if degenerate_boxes.any():
                    bb_idx = torch.where(degenerate_boxes.any(dim=1))[0][0]
                    raise ValueError(
                        f"Bounding box tidak valid pada indeks {bb_idx}"
                    )

    original_image_sizes = []

    for img in images:
        val = img.shape[-2:]
        original_image_sizes.append((val[0], val[1]))

    images, targets = self.transform(images, targets)

    features = self.backbone(images.tensors)

    if isinstance(features, torch.Tensor):
        features = OrderedDict([("0", features)])

    proposals, proposal_losses = self.rpn(
        images, features, targets
    )

    detections, detector_losses = self.roi_heads(
        features, proposals, images.image_sizes, targets
    )

    detections = self.transform.postprocess(
        detections, images.image_sizes, original_image_sizes
    )

    losses = {}
    losses.update(detector_losses)
    losses.update(proposal_losses)

    return self.eager_outputs(losses, detections)


model.forward = MethodType(forward_safe, model)


# ==================================================
# 3. PATCH ROI HEADS
# Menangani GT kosong ketika menambahkan proposal
# ==================================================

def add_gt_proposals_safe(self, proposals, gt_boxes):

    updated_proposals = []

    for proposal, gt_box in zip(proposals, gt_boxes):

        if gt_box.numel() == 0:
            updated_proposals.append(proposal)
        else:
            updated_proposals.append(
                torch.cat((proposal, gt_box), dim=0)
            )

    return updated_proposals


model.roi_heads.add_gt_proposals = MethodType(
    add_gt_proposals_safe,
    model.roi_heads
)


# ==================================================
# 4. PATCH FAST RCNN LOSS
# Menangani kondisi tidak ada proposal positif
# ==================================================

def fastrcnn_loss_safe(
    class_logits,
    box_regression,
    labels,
    regression_targets
):

    labels = torch.cat(labels, dim=0)
    regression_targets = torch.cat(regression_targets, dim=0)

    classification_loss = F.cross_entropy(class_logits, labels)

    sampled_pos_inds = torch.where(labels > 0)[0]
    labels_pos = labels[sampled_pos_inds]

    N, num_classes = class_logits.shape

    box_regression = box_regression.reshape(
        N, box_regression.size(-1) // 4, 4
    )

    if sampled_pos_inds.numel() == 0:
        box_loss = box_regression.sum() * 0.0
    else:
        box_loss = F.smooth_l1_loss(
            box_regression[sampled_pos_inds, labels_pos],
            regression_targets[sampled_pos_inds],
            beta=1 / 9,
            reduction="sum"
        )

    box_loss = box_loss / max(labels.numel(), 1)

    return classification_loss, box_loss


roi_heads_module.fastrcnn_loss = fastrcnn_loss_safe


print("Seluruh patch berhasil dipasang kembali.")

Seluruh patch berhasil dipasang kembali.


In [40]:
# Forward gambar problematik alias kosong
# Ambil gambar training yang sebelumnya diketahui tidak memiliki bounding box
empty_index = 381

image_empty, target_empty = train_dataset[empty_index]

print("Nama gambar:", train_dataset.images[empty_index])
print("Jumlah bounding box:", len(target_empty["boxes"]))

images_empty = [image_empty.to(device)]

targets_empty = [{
    key: value.to(device) if isinstance(value, torch.Tensor) else value
    for key, value in target_empty.items()
}]

model.train()

with torch.no_grad():
    loss_dict_empty = model(images_empty, targets_empty)

print("\nLoss pada gambar tanpa bounding box:")

for name, loss in loss_dict_empty.items():
    print(f"{name}: {loss.item():.6f}")

print("Total loss:", sum(loss_dict_empty.values()).item())

Nama gambar: C:\Faster-RCNN-PPE\dataset\images\train\ppe_0169_jpg.rf.d85b8729987cec65187b118f9e8c6328.jpg
Jumlah bounding box: 0

Loss pada gambar tanpa bounding box:
loss_classifier: 2.382572
loss_box_reg: 0.000000
loss_objectness: 0.005289
loss_rpn_box_reg: 0.000000
Total loss: 2.3878602981567383


In [41]:
# Forward gambar dengan APD
sample_index = 0

image_positive, target_positive = train_dataset[sample_index]

print("Nama gambar:", train_dataset.images[sample_index])
print("Jumlah bounding box:", len(target_positive["boxes"]))

images_positive = [image_positive.to(device)]

targets_positive = [{
    key: value.to(device) if isinstance(value, torch.Tensor) else value
    for key, value in target_positive.items()
}]

model.train()

with torch.no_grad():
    loss_dict_positive = model(images_positive, targets_positive)

print("\nLoss pada gambar dengan objek:")

for name, loss in loss_dict_positive.items():
    print(f"{name}: {loss.item():.6f}")

print("Total loss:", sum(loss_dict_positive.values()).item())

Nama gambar: C:\Faster-RCNN-PPE\dataset\images\train\000004_jpg.rf.a64a312cbd6f217ceb7680f99531fcd6.jpg
Jumlah bounding box: 11

Loss pada gambar dengan objek:
loss_classifier: 2.544379
loss_box_reg: 0.920051
loss_objectness: 0.033150
loss_rpn_box_reg: 0.088732
Total loss: 3.5863120555877686


In [42]:
# Uji backward & optimizer
model.train()
optimizer.zero_grad()

loss_dict_positive = model(images_positive, targets_positive)
total_loss_positive = sum(loss_dict_positive.values())

print(f"Total loss sebelum backward: {total_loss_positive.item():.6f}")

total_loss_positive.backward()
print("Backward berhasil.")

optimizer.step()
print("Optimizer step berhasil.")

Total loss sebelum backward: 3.633785
Backward berhasil.
Optimizer step berhasil.


In [43]:
# Inisialisasi final untuk training 1 epoch
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

weights = torchvision.models.detection.FasterRCNN_ResNet50_FPN_Weights.DEFAULT

model = fasterrcnn_resnet50_fpn(
    weights=weights,
    min_size=640,
    max_size=640
)

in_features = model.roi_heads.box_predictor.cls_score.in_features

model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
    num_classes=9
)

model.to(device)

optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=0.001,
    momentum=0.9,
    weight_decay=0.0005
)

print("Model final untuk uji 1 epoch berhasil diinisialisasi.")
print("Device:", next(model.parameters()).device)
print("Jumlah kelas:", model.roi_heads.box_predictor.cls_score.out_features)

Model final untuk uji 1 epoch berhasil diinisialisasi.
Device: privateuseone:0
Jumlah kelas: 9


In [44]:
# Periksa konfigurasi model
print("=== PEMERIKSAAN MODEL FINAL ===")

print("Device model:", next(model.parameters()).device)
print("Jumlah kelas:", model.roi_heads.box_predictor.cls_score.out_features)

print("Patch RPN:",
      model.rpn.compute_loss.__func__.__name__)

print("Patch Forward:",
      model.forward.__func__.__name__)

print("Patch ROI:",
      model.roi_heads.add_gt_proposals.__func__.__name__)

print("Patch Fast R-CNN:",
      roi_heads_module.fastrcnn_loss.__name__)

print("\nJumlah data training:", len(train_dataset))
print("Jumlah batch training:", len(train_loader))
print("Optimizer:", type(optimizer).__name__)

=== PEMERIKSAAN MODEL FINAL ===
Device model: privateuseone:0
Jumlah kelas: 9
Patch RPN: compute_loss
Patch Forward: forward
Patch ROI: add_gt_proposals
Patch Fast R-CNN: fastrcnn_loss_safe

Jumlah data training: 1189
Jumlah batch training: 1189
Optimizer: SGD


In [45]:
# Buat folder simpan checkpoint epoch 1
import os
import time
import json

# Folder khusus eksperimen 1 epoch
output_dir = r"C:\Faster-RCNN-PPE\experiments\fasterrcnn_1_epoch"

os.makedirs(output_dir, exist_ok=True)

# Konfigurasi eksperimen
num_epochs = 1
learning_rate = 0.001

# Lokasi checkpoint
checkpoint_path = os.path.join(
    output_dir,
    "fasterrcnn_epoch_checkpoint.pth"
)

# Lokasi log training
log_path = os.path.join(
    output_dir,
    "training_log.json"
)

print("Folder eksperimen:", output_dir)
print("Checkpoint:", checkpoint_path)
print("Log:", log_path)
print("Jumlah epoch:", num_epochs)
print("Learning rate:", learning_rate)

print("\nFolder berhasil disiapkan.")

Folder eksperimen: C:\Faster-RCNN-PPE\experiments\fasterrcnn_1_epoch
Checkpoint: C:\Faster-RCNN-PPE\experiments\fasterrcnn_1_epoch\fasterrcnn_epoch_checkpoint.pth
Log: C:\Faster-RCNN-PPE\experiments\fasterrcnn_1_epoch\training_log.json
Jumlah epoch: 1
Learning rate: 0.001

Folder berhasil disiapkan.


In [46]:
# Training 1 epoch
import time
import json
import os
import torch

model.train()

num_epochs = 1
log_interval = 100
checkpoint_interval = 100

epoch_losses = []
running_loss = 0.0
training_log = []

start_time = time.time()

print("=== MEMULAI TRAINING FASTER R-CNN ===")

for epoch in range(num_epochs):

    epoch_start = time.time()
    total_epoch_loss = 0.0

    for batch_idx, (images, targets) in enumerate(train_loader):

        images = [image.to(device) for image in images]

        targets = [
            {
                key: value.to(device) if isinstance(value, torch.Tensor) else value
                for key, value in target.items()
            }
            for target in targets
        ]

        optimizer.zero_grad()

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        loss_value = losses.item()

        if not torch.isfinite(losses):
            raise RuntimeError(
                f"Loss tidak valid pada batch {batch_idx + 1}: {loss_value}"
            )

        losses.backward()
        optimizer.step()

        total_epoch_loss += loss_value
        running_loss += loss_value

        if (batch_idx + 1) % log_interval == 0:

            average_loss = running_loss / log_interval

            print(
                f"Epoch [{epoch + 1}/{num_epochs}] "
                f"Batch [{batch_idx + 1}/{len(train_loader)}] "
                f"Average Loss: {average_loss:.6f}"
            )

            training_log.append({
                "epoch": epoch + 1,
                "batch": batch_idx + 1,
                "average_loss": average_loss
            })

            running_loss = 0.0

        # Simpan checkpoint berkala
        if (batch_idx + 1) % checkpoint_interval == 0:

            checkpoint = {
                "epoch": epoch + 1,
                "batch": batch_idx + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "total_epoch_loss": total_epoch_loss
            }

            torch.save(checkpoint, checkpoint_path)

            print(f"Checkpoint tersimpan pada batch {batch_idx + 1}")

    epoch_duration = time.time() - epoch_start
    average_epoch_loss = total_epoch_loss / len(train_loader)

    epoch_losses.append(average_epoch_loss)

    print("\n=== HASIL EPOCH ===")
    print(f"Epoch: {epoch + 1}")
    print(f"Average Training Loss: {average_epoch_loss:.6f}")
    print(f"Durasi: {epoch_duration / 60:.2f} menit")

    # Simpan checkpoint akhir
    checkpoint = {
        "epoch": epoch + 1,
        "batch": len(train_loader),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "average_epoch_loss": average_epoch_loss
    }

    torch.save(checkpoint, checkpoint_path)

# Simpan log
training_result = {
    "epochs": num_epochs,
    "total_images": len(train_dataset),
    "batch_size": train_loader.batch_size,
    "average_training_loss": epoch_losses,
    "training_duration_seconds": time.time() - start_time,
    "batch_logs": training_log
}

with open(log_path, "w") as file:
    json.dump(training_result, file, indent=4)

print("\n=== TRAINING SELESAI ===")
print("Checkpoint:", checkpoint_path)
print("Log:", log_path)

=== MEMULAI TRAINING FASTER R-CNN ===
Epoch [1/1] Batch [100/1189] Average Loss: 1.163663
Checkpoint tersimpan pada batch 100
Epoch [1/1] Batch [200/1189] Average Loss: 0.817403
Checkpoint tersimpan pada batch 200


RuntimeError: output with shape [] doesn't match the broadcast shape [0, 4]

In [47]:
print("=== STATUS PATCH PADA MODEL ===")

print(
    "RPN:",
    model.rpn.compute_loss.__func__.__name__
    if hasattr(model.rpn.compute_loss, "__func__")
    else model.rpn.compute_loss.__name__
)

print(
    "Forward:",
    model.forward.__func__.__name__
    if hasattr(model.forward, "__func__")
    else model.forward.__name__
)

print(
    "ROI:",
    model.roi_heads.add_gt_proposals.__func__.__name__
    if hasattr(model.roi_heads.add_gt_proposals, "__func__")
    else model.roi_heads.add_gt_proposals.__name__
)

print(
    "Fast R-CNN:",
    roi_heads_module.fastrcnn_loss.__name__
)

print("\n=== STATUS CHECKPOINT ===")
print("Checkpoint tersedia:", os.path.exists(checkpoint_path))

=== STATUS PATCH PADA MODEL ===
RPN: compute_loss
Forward: forward
ROI: add_gt_proposals
Fast R-CNN: fastrcnn_loss_safe

=== STATUS CHECKPOINT ===
Checkpoint tersedia: True


In [48]:
checkpoint_data = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("=== INFORMASI CHECKPOINT ===")
print("Epoch:", checkpoint_data["epoch"])
print("Batch terakhir:", checkpoint_data["batch"])
print("Keys:", list(checkpoint_data.keys()))

print(
    "Total epoch loss sementara:",
    checkpoint_data.get("total_epoch_loss", "Tidak tersedia")
)

=== INFORMASI CHECKPOINT ===
Epoch: 1
Batch terakhir: 200
Keys: ['epoch', 'batch', 'model_state_dict', 'optimizer_state_dict', 'total_epoch_loss']
Total epoch loss sementara: 198.10662478208542


In [49]:
# Pulihkan bobot model dari checkpoint batch 200
model.load_state_dict(checkpoint_data["model_state_dict"])

# Pulihkan kondisi optimizer, termasuk momentum SGD
optimizer.load_state_dict(checkpoint_data["optimizer_state_dict"])

# Pastikan model kembali dalam mode training
model.train()

print("=== PEMULIHAN CHECKPOINT ===")
print("Epoch:", checkpoint_data["epoch"])
print("Batch terakhir:", checkpoint_data["batch"])
print("Total loss sementara:", checkpoint_data["total_epoch_loss"])
print("Device model:", next(model.parameters()).device)
print("Optimizer:", type(optimizer).__name__)
print("Pemulihan berhasil.")

=== PEMULIHAN CHECKPOINT ===
Epoch: 1
Batch terakhir: 200
Total loss sementara: 198.10662478208542
Device model: privateuseone:0
Optimizer: SGD
Pemulihan berhasil.


In [50]:
import torch
import torch.nn.functional as F
from types import MethodType
from collections import OrderedDict
import torchvision.models.detection.roi_heads as roi_heads_module


# 1. Patch RPN: menangani kondisi tidak ada positive anchor
def compute_loss_safe(self, objectness, pred_bbox_deltas, labels, regression_targets):
    sampled_pos_inds, sampled_neg_inds = self.fg_bg_sampler(labels)

    sampled_pos_inds = torch.where(torch.cat(sampled_pos_inds, dim=0))[0]
    sampled_neg_inds = torch.where(torch.cat(sampled_neg_inds, dim=0))[0]
    sampled_inds = torch.cat([sampled_pos_inds, sampled_neg_inds])

    objectness = objectness.flatten()
    labels = torch.cat(labels, dim=0)
    regression_targets = torch.cat(regression_targets, dim=0)

    if sampled_pos_inds.numel() == 0:
        box_loss = pred_bbox_deltas.sum() * 0.0
    else:
        box_loss = F.smooth_l1_loss(
            pred_bbox_deltas[sampled_pos_inds],
            regression_targets[sampled_pos_inds],
            beta=1 / 9,
            reduction="sum"
        )

    box_loss = box_loss / max(sampled_inds.numel(), 1)

    if sampled_inds.numel() == 0:
        objectness_loss = objectness.sum() * 0.0
    else:
        objectness_loss = F.binary_cross_entropy_with_logits(
            objectness[sampled_inds],
            labels[sampled_inds]
        )

    return objectness_loss, box_loss


# 2. Patch forward: melewati pemeriksaan bounding box jika kosong
def forward_safe(self, images, targets=None):
    if self.training and targets is None:
        raise ValueError("Targets harus diberikan saat training.")

    if self.training:
        for target in targets:
            boxes = target["boxes"]

            if isinstance(boxes, torch.Tensor) and boxes.numel() > 0:
                degenerate_boxes = boxes[:, 2:] <= boxes[:, :2]

                if degenerate_boxes.any():
                    bb_idx = torch.where(degenerate_boxes.any(dim=1))[0][0]
                    raise ValueError(
                        f"Bounding box tidak valid pada indeks {bb_idx}"
                    )

    original_image_sizes = []

    for img in images:
        val = img.shape[-2:]
        original_image_sizes.append((val[0], val[1]))

    images, targets = self.transform(images, targets)

    features = self.backbone(images.tensors)

    if isinstance(features, torch.Tensor):
        features = OrderedDict([("0", features)])

    proposals, proposal_losses = self.rpn(images, features, targets)

    detections, detector_losses = self.roi_heads(
        features, proposals, images.image_sizes, targets
    )

    detections = self.transform.postprocess(
        detections, images.image_sizes, original_image_sizes
    )

    losses = {}
    losses.update(detector_losses)
    losses.update(proposal_losses)

    return self.eager_outputs(losses, detections)


# 3. Patch ROI Heads: tidak menambahkan GT proposal jika kosong
def add_gt_proposals_safe(self, proposals, gt_boxes):
    updated_proposals = []

    for proposal, gt_box in zip(proposals, gt_boxes):
        if gt_box.numel() == 0:
            updated_proposals.append(proposal)
        else:
            updated_proposals.append(torch.cat((proposal, gt_box), dim=0))

    return updated_proposals


# 4. Patch Fast R-CNN: menangani kondisi tidak ada positive proposal
def fastrcnn_loss_safe(class_logits, box_regression, labels, regression_targets):
    labels = torch.cat(labels, dim=0)
    regression_targets = torch.cat(regression_targets, dim=0)

    classification_loss = F.cross_entropy(class_logits, labels)

    sampled_pos_inds = torch.where(labels > 0)[0]
    labels_pos = labels[sampled_pos_inds]

    N, num_classes = class_logits.shape
    box_regression = box_regression.reshape(
        N, box_regression.size(-1) // 4, 4
    )

    if sampled_pos_inds.numel() == 0:
        box_loss = box_regression.sum() * 0.0
    else:
        box_loss = F.smooth_l1_loss(
            box_regression[sampled_pos_inds, labels_pos],
            regression_targets[sampled_pos_inds],
            beta=1 / 9,
            reduction="sum"
        )

    box_loss = box_loss / max(labels.numel(), 1)

    return classification_loss, box_loss


# Pasang semua patch pada model yang sedang digunakan
model.rpn.compute_loss = MethodType(compute_loss_safe, model.rpn)
model.forward = MethodType(forward_safe, model)
model.roi_heads.add_gt_proposals = MethodType(
    add_gt_proposals_safe, model.roi_heads
)
roi_heads_module.fastrcnn_loss = fastrcnn_loss_safe

print("Semua patch DirectML telah dipasang.")

Semua patch DirectML telah dipasang.


In [51]:
print("=== VERIFIKASI PATCH DIRECTML ===")

print("1. RPN          :", model.rpn.compute_loss.__func__.__name__)
print("2. Model Forward:", model.forward.__func__.__name__)
print("3. ROI Heads    :", model.roi_heads.add_gt_proposals.__func__.__name__)
print("4. Fast R-CNN   :", roi_heads_module.fastrcnn_loss.__name__)

=== VERIFIKASI PATCH DIRECTML ===
1. RPN          : compute_loss_safe
2. Model Forward: forward_safe
3. ROI Heads    : add_gt_proposals_safe
4. Fast R-CNN   : fastrcnn_loss_safe


In [52]:
# Uji forward pada gambar problematik
# Ambil gambar background yang tidak memiliki bounding box
image, target = train_dataset[381]

print("Jumlah bounding box:", len(target["boxes"]))

# Pindahkan data ke DirectML
image = image.to(device)

target = {
    key: value.to(device) if isinstance(value, torch.Tensor) else value
    for key, value in target.items()
}

# Uji forward pass tanpa memperbarui bobot model
model.train()

with torch.no_grad():
    losses = model([image], [target])

print("\n=== HASIL UJI FORWARD BACKGROUND ===")

for name, value in losses.items():
    print(f"{name}: {value.item():.6f}")

total_loss = sum(losses.values())
print(f"Total loss: {total_loss.item():.6f}")
print("Uji forward pass berhasil.")

Jumlah bounding box: 0

=== HASIL UJI FORWARD BACKGROUND ===
loss_classifier: 0.041288
loss_box_reg: 0.000000
loss_objectness: 0.039710
loss_rpn_box_reg: 0.000000
Total loss: 0.080998
Uji forward pass berhasil.


In [53]:
# Pulihkan kembali kondisi checkpoint batch 200 setelah pengujian
model.load_state_dict(checkpoint_data["model_state_dict"])
optimizer.load_state_dict(checkpoint_data["optimizer_state_dict"])
model.train()

print("Checkpoint dipulihkan kembali.")
print("Epoch:", checkpoint_data["epoch"])
print("Batch:", checkpoint_data["batch"])
print("Sampler:", type(train_loader.sampler).__name__)
print("Shuffle:", isinstance(train_loader.sampler, torch.utils.data.RandomSampler))

Checkpoint dipulihkan kembali.
Epoch: 1
Batch: 200
Sampler: RandomSampler
Shuffle: True


In [54]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from types import MethodType
from collections import OrderedDict
import torch.nn.functional as F
import torchvision.models.detection.roi_heads as roi_heads_module

# Inisialisasi model baru dengan pretrained COCO weights
model = fasterrcnn_resnet50_fpn(
    weights=torchvision.models.detection.FasterRCNN_ResNet50_FPN_Weights.DEFAULT,
    min_size=640,
    max_size=640
)

# Sesuaikan predictor dengan 8 kelas APD + background
in_features = model.roi_heads.box_predictor.cls_score.in_features
num_classes = 9

model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
    num_classes
)

# Pindahkan model ke DirectML
model.to(device)

# Buat optimizer baru
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.001,
    momentum=0.9,
    weight_decay=0.0005
)

print("Model baru berhasil diinisialisasi.")
print("Jumlah kelas:", num_classes)
print("Device:", next(model.parameters()).device)
print("Optimizer: SGD")

Model baru berhasil diinisialisasi.
Jumlah kelas: 9
Device: privateuseone:0
Optimizer: SGD


In [55]:
# 1. RPN
model.rpn.compute_loss = MethodType(compute_loss_safe, model.rpn)

# 2. Model Forward
model.forward = MethodType(forward_safe, model)

# 3. ROI Heads
model.roi_heads.add_gt_proposals = MethodType(
    add_gt_proposals_safe, model.roi_heads
)

# 4. Fast R-CNN
roi_heads_module.fastrcnn_loss = fastrcnn_loss_safe

print("=== VERIFIKASI PATCH MODEL BARU ===")
print("RPN          :", model.rpn.compute_loss.__func__.__name__)
print("Model Forward:", model.forward.__func__.__name__)
print("ROI Heads    :", model.roi_heads.add_gt_proposals.__func__.__name__)
print("Fast R-CNN   :", roi_heads_module.fastrcnn_loss.__name__)

=== VERIFIKASI PATCH MODEL BARU ===
RPN          : compute_loss_safe
Model Forward: forward_safe
ROI Heads    : add_gt_proposals_safe
Fast R-CNN   : fastrcnn_loss_safe


In [56]:
# Training 1 epoch
import time

# Pastikan model dalam mode training
model.train()

# Variabel pencatatan loss
total_epoch_loss = 0.0
running_loss = 0.0
training_log = []

num_batches = len(train_loader)

print("=== KONFIGURASI TRAINING FASTER R-CNN ===")
print("Jumlah data training :", len(train_dataset))
print("Jumlah batch         :", num_batches)
print("Batch size           :", train_loader.batch_size)
print("Epoch                : 1")
print("Learning rate        :", optimizer.param_groups[0]["lr"])
print("Device               :", next(model.parameters()).device)
print("Penyimpanan checkpoint: Tidak")
print("Penyimpanan log       : Tidak")
print("\nModel siap dilatih.")

=== KONFIGURASI TRAINING FASTER R-CNN ===
Jumlah data training : 1189
Jumlah batch         : 1189
Batch size           : 1
Epoch                : 1
Learning rate        : 0.001
Device               : privateuseone:0
Penyimpanan checkpoint: Tidak
Penyimpanan log       : Tidak

Model siap dilatih.


In [57]:
# eksekusi training
import time

model.train()

total_epoch_loss = 0.0
running_loss = 0.0
training_log = []

start_time = time.time()

for batch_idx, (images, targets) in enumerate(train_loader):

    # Pindahkan gambar dan target ke DirectML
    images = [image.to(device) for image in images]

    targets = [
        {
            key: value.to(device) if isinstance(value, torch.Tensor) else value
            for key, value in target.items()
        }
        for target in targets
    ]

    # Reset gradient
    optimizer.zero_grad()

    # Forward pass
    loss_dict = model(images, targets)

    # Hitung total loss
    losses = sum(loss for loss in loss_dict.values())

    # Backward pass
    losses.backward()

    # Update parameter
    optimizer.step()

    batch_loss = losses.item()

    total_epoch_loss += batch_loss
    running_loss += batch_loss

    # Tampilkan progres setiap 100 batch
    if (batch_idx + 1) % 100 == 0 or (batch_idx + 1) == num_batches:

        avg_loss = running_loss / (
            100 if (batch_idx + 1) % 100 == 0 else
            ((batch_idx + 1) % 100)
        )

        elapsed = time.time() - start_time

        print(
            f"Batch [{batch_idx + 1}/{num_batches}] | "
            f"Average Loss: {avg_loss:.6f} | "
            f"Elapsed: {elapsed / 60:.2f} menit"
        )

        training_log.append({
            "batch": batch_idx + 1,
            "average_loss": avg_loss
        })

        running_loss = 0.0

# Rata-rata loss seluruh epoch
average_epoch_loss = total_epoch_loss / num_batches

total_time = time.time() - start_time

print("\n=== TRAINING SELESAI ===")
print("Epoch              : 1")
print("Total batch        :", num_batches)
print(f"Total loss         : {total_epoch_loss:.6f}")
print(f"Average epoch loss : {average_epoch_loss:.6f}")
print(f"Waktu training     : {total_time / 60:.2f} menit")

Batch [100/1189] | Average Loss: 1.192760 | Elapsed: 3.38 menit
Batch [200/1189] | Average Loss: 0.866612 | Elapsed: 6.75 menit
Batch [300/1189] | Average Loss: 0.711653 | Elapsed: 10.09 menit
Batch [400/1189] | Average Loss: 0.692260 | Elapsed: 13.46 menit
Batch [500/1189] | Average Loss: 0.638000 | Elapsed: 16.85 menit
Batch [600/1189] | Average Loss: 0.603473 | Elapsed: 20.23 menit
Batch [700/1189] | Average Loss: 0.599891 | Elapsed: 23.64 menit
Batch [800/1189] | Average Loss: 0.562521 | Elapsed: 28.35 menit
Batch [900/1189] | Average Loss: 0.489518 | Elapsed: 31.80 menit
Batch [1000/1189] | Average Loss: 0.509927 | Elapsed: 35.35 menit
Batch [1100/1189] | Average Loss: 0.482476 | Elapsed: 38.81 menit
Batch [1189/1189] | Average Loss: 0.498382 | Elapsed: 41.89 menit

=== TRAINING SELESAI ===
Epoch              : 1
Total batch        : 1189
Total loss         : 779.265172
Average epoch loss : 0.655395
Waktu training     : 41.89 menit


In [58]:
import os
import torch

# Folder khusus eksperimen 20 epoch
output_dir = r"C:\Faster-RCNN-PPE\experiments\fasterrcnn_20_epochs"
os.makedirs(output_dir, exist_ok=True)

# Fungsi untuk mengubah tensor menjadi CPU sebelum disimpan
def move_to_cpu(data):
    if isinstance(data, torch.Tensor):
        return data.detach().cpu()
    elif isinstance(data, dict):
        return {key: move_to_cpu(value) for key, value in data.items()}
    elif isinstance(data, list):
        return [move_to_cpu(value) for value in data]
    elif isinstance(data, tuple):
        return tuple(move_to_cpu(value) for value in data)
    return data

# Siapkan checkpoint epoch pertama
checkpoint = {
    "epoch": 1,
    "model_state_dict": move_to_cpu(model.state_dict()),
    "optimizer_state_dict": move_to_cpu(optimizer.state_dict()),
    "average_epoch_loss": average_epoch_loss,
    "total_epoch_loss": total_epoch_loss,
    "num_classes": 9,
    "learning_rate": 0.001
}

# Simpan checkpoint
checkpoint_path = os.path.join(
    output_dir,
    "fasterrcnn_epoch_1.pth"
)

torch.save(checkpoint, checkpoint_path)

print("=== CHECKPOINT EPOCH 1 ===")
print("Lokasi:", checkpoint_path)
print("Epoch:", checkpoint["epoch"])
print(f"Average loss: {average_epoch_loss:.6f}")
print(f"Ukuran file: {os.path.getsize(checkpoint_path) / (1024**2):.2f} MB")
print("Checkpoint berhasil disimpan.")

=== CHECKPOINT EPOCH 1 ===
Lokasi: C:\Faster-RCNN-PPE\experiments\fasterrcnn_20_epochs\fasterrcnn_epoch_1.pth
Epoch: 1
Average loss: 0.655395
Ukuran file: 315.03 MB
Checkpoint berhasil disimpan.


In [59]:
# periksa kondisi model
import os
import torch

print("=== PEMERIKSAAN MODEL ===")
print("Device:", next(model.parameters()).device)
print("Mode training:", model.training)
print("Optimizer:", type(optimizer).__name__)
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Data training:", len(train_dataset))
print("Batch per epoch:", len(train_loader))

print("\n=== PEMERIKSAAN PATCH ===")
print("RPN:", model.rpn.compute_loss.__func__.__name__)
print("Forward:", model.forward.__func__.__name__)
print("ROI Heads:", model.roi_heads.add_gt_proposals.__func__.__name__)
print("Fast R-CNN:", roi_heads_module.fastrcnn_loss.__name__)

print("\n=== CHECKPOINT ===")
print("Checkpoint epoch 1:", os.path.exists(checkpoint_path))
print("Folder eksperimen:", output_dir)

=== PEMERIKSAAN MODEL ===
Device: privateuseone:0
Mode training: True
Optimizer: SGD
Learning rate: 0.001
Data training: 1189
Batch per epoch: 1189

=== PEMERIKSAAN PATCH ===
RPN: compute_loss_safe
Forward: forward_safe
ROI Heads: add_gt_proposals_safe
Fast R-CNN: fastrcnn_loss_safe

=== CHECKPOINT ===
Checkpoint epoch 1: True
Folder eksperimen: C:\Faster-RCNN-PPE\experiments\fasterrcnn_20_epochs


In [60]:
# Lanjut training sampai epoch 5
import os
import time
import torch

def move_to_cpu(data):
    if isinstance(data, torch.Tensor):
        return data.detach().cpu()
    elif isinstance(data, dict):
        return {k: move_to_cpu(v) for k, v in data.items()}
    elif isinstance(data, list):
        return [move_to_cpu(v) for v in data]
    elif isinstance(data, tuple):
        return tuple(move_to_cpu(v) for v in data)
    return data


training_history = []

for epoch in range(2, 6):

    model.train()

    total_loss = 0.0
    start_time = time.time()

    print(f"\n{'='*55}")
    print(f"MEMULAI TRAINING EPOCH {epoch}/20")
    print(f"{'='*55}")

    for batch_idx, (images, targets) in enumerate(train_loader):

        images = [img.to(device) for img in images]

        targets = [
            {k: v.to(device) for k, v in target.items()}
            for target in targets
        ]

        optimizer.zero_grad()

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        losses.backward()
        optimizer.step()

        batch_loss = losses.item()
        total_loss += batch_loss

        if (batch_idx + 1) % 100 == 0 or (batch_idx + 1) == len(train_loader):
            print(
                f"Epoch {epoch} | "
                f"Batch {batch_idx + 1}/{len(train_loader)} | "
                f"Loss: {batch_loss:.6f} | "
                f"Rata-rata loss: {total_loss / (batch_idx + 1):.6f}"
            )

    average_loss = total_loss / len(train_loader)
    elapsed_time = (time.time() - start_time) / 60

    training_history.append({
        "epoch": epoch,
        "total_loss": total_loss,
        "average_loss": average_loss,
        "time_minutes": elapsed_time
    })

    print(f"\nHASIL EPOCH {epoch}")
    print(f"Total loss       : {total_loss:.6f}")
    print(f"Average loss     : {average_loss:.6f}")
    print(f"Waktu training   : {elapsed_time:.2f} menit")

    # Simpan checkpoint terbaru setiap epoch
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": move_to_cpu(model.state_dict()),
        "optimizer_state_dict": move_to_cpu(optimizer.state_dict()),
        "average_epoch_loss": average_loss,
        "total_epoch_loss": total_loss,
        "num_classes": 9,
        "learning_rate": optimizer.param_groups[0]["lr"]
    }

    latest_path = os.path.join(output_dir, "fasterrcnn_latest.pth")
    temp_path = latest_path + ".tmp"

    torch.save(checkpoint, temp_path)
    os.replace(temp_path, latest_path)

    print(f"Checkpoint terbaru tersimpan: {latest_path}")

    # Simpan checkpoint khusus pada epoch kelipatan 5
    if epoch == 5:
        milestone_path = os.path.join(
            output_dir, "fasterrcnn_epoch_5.pth"
        )
        torch.save(checkpoint, milestone_path)
        print(f"Checkpoint milestone tersimpan: {milestone_path}")

    print(f"Epoch {epoch} selesai.")

print("\nTRAINING TAHAP 2–5 SELESAI.")


MEMULAI TRAINING EPOCH 2/20
Epoch 2 | Batch 100/1189 | Loss: 0.982640 | Rata-rata loss: 0.431590
Epoch 2 | Batch 200/1189 | Loss: 0.138381 | Rata-rata loss: 0.452774
Epoch 2 | Batch 300/1189 | Loss: 0.534580 | Rata-rata loss: 0.442063
Epoch 2 | Batch 400/1189 | Loss: 0.272407 | Rata-rata loss: 0.442151
Epoch 2 | Batch 500/1189 | Loss: 0.503687 | Rata-rata loss: 0.440255
Epoch 2 | Batch 600/1189 | Loss: 0.355212 | Rata-rata loss: 0.437260
Epoch 2 | Batch 700/1189 | Loss: 0.211079 | Rata-rata loss: 0.436142
Epoch 2 | Batch 800/1189 | Loss: 0.551988 | Rata-rata loss: 0.432367
Epoch 2 | Batch 900/1189 | Loss: 0.247289 | Rata-rata loss: 0.434000
Epoch 2 | Batch 1000/1189 | Loss: 0.268337 | Rata-rata loss: 0.433388
Epoch 2 | Batch 1100/1189 | Loss: 0.200655 | Rata-rata loss: 0.434647
Epoch 2 | Batch 1189/1189 | Loss: 0.274667 | Rata-rata loss: 0.433468

HASIL EPOCH 2
Total loss       : 515.393648
Average loss     : 0.433468
Waktu training   : 42.05 menit
Checkpoint terbaru tersimpan: C:\Fas

In [1]:
# Aktifkan kembali environment DirectML
import os
import torch
import torchvision
import torch_directml

checkpoint_path = (
    r"C:\Faster-RCNN-PPE\experiments\fasterrcnn_20_epochs"
    r"\fasterrcnn_epoch_5.pth"
)

device = torch_directml.device()

print("=== PEMERIKSAAN ENVIRONMENT ===")
print("PyTorch       :", torch.__version__)
print("Torchvision   :", torchvision.__version__)
print("DirectML      :", device)

print("\n=== PEMERIKSAAN CHECKPOINT ===")
print("Checkpoint ditemukan:", os.path.exists(checkpoint_path))

if os.path.exists(checkpoint_path):
    print(
        "Ukuran checkpoint:",
        round(os.path.getsize(checkpoint_path) / (1024**2), 2),
        "MB"
    )

=== PEMERIKSAAN ENVIRONMENT ===
PyTorch       : 2.4.1+cpu
Torchvision   : 0.19.1+cpu
DirectML      : privateuseone:0

=== PEMERIKSAAN CHECKPOINT ===
Checkpoint ditemukan: True
Ukuran checkpoint: 315.03 MB


In [2]:
# Pulihkan dataset dan dataLoader
import os
import torch
import xml.etree.ElementTree as ET

from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import to_tensor

# Lokasi dataset
ROOT = Path(r"C:\Faster-RCNN-PPE\dataset\images")

# Mapping kelas
CLASS_MAP = {
    "person": 1,
    "vest": 2,
    "yellow": 3,
    "white": 4,
    "glass": 5,
    "blue": 6,
    "red": 7,
    "head": 8
}


class CHVGPpeDataset(Dataset):

    def __init__(self, root, split):
        self.folder = Path(root) / split

        self.image_paths = sorted(
            list(self.folder.glob("*.jpg")) +
            list(self.folder.glob("*.jpeg"))
        )

        if not self.image_paths:
            raise RuntimeError(
                f"Tidak ditemukan gambar di {self.folder}"
            )

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        image_path = self.image_paths[idx]
        xml_path = self.folder / (image_path.stem + ".xml")

        image = Image.open(image_path).convert("RGB")
        width, height = image.size

        tree = ET.parse(xml_path)
        root = tree.getroot()

        boxes = []
        labels = []

        for obj in root.findall("object"):

            class_name = obj.find("name").text.strip()

            if class_name not in CLASS_MAP:
                raise ValueError(
                    f"Kelas tidak dikenal: {class_name}"
                )

            bbox = obj.find("bndbox")

            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)

            # Clipping bounding box
            xmin = max(0, min(xmin, width))
            ymin = max(0, min(ymin, height))
            xmax = max(0, min(xmax, width))
            ymax = max(0, min(ymax, height))

            # Lewati bounding box tidak valid
            if xmax <= xmin or ymax <= ymin:
                continue

            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(CLASS_MAP[class_name])

        boxes = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": (
                (boxes[:, 3] - boxes[:, 1]) *
                (boxes[:, 2] - boxes[:, 0])
            ),
            "iscrowd": torch.zeros(
                (len(boxes),), dtype=torch.int64
            )
        }

        return to_tensor(image), target


# Membuat dataset
train_dataset = CHVGPpeDataset(ROOT, "train")
val_dataset = CHVGPpeDataset(ROOT, "val")
test_dataset = CHVGPpeDataset(ROOT, "test")


# Fungsi collate
def collate_fn(batch):
    return tuple(zip(*batch))


# Membuat DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)


print("=== PEMERIKSAAN DATASET ===")
print("Train dataset :", len(train_dataset))
print("Val dataset   :", len(val_dataset))
print("Test dataset  :", len(test_dataset))

print("\n=== PEMERIKSAAN DATALOADER ===")
print("Train batches :", len(train_loader))
print("Val batches   :", len(val_loader))

print("\nDataset dan DataLoader berhasil dibuat.")

=== PEMERIKSAAN DATASET ===
Train dataset : 1189
Val dataset   : 170
Test dataset  : 340

=== PEMERIKSAAN DATALOADER ===
Train batches : 1189
Val batches   : 170

Dataset dan DataLoader berhasil dibuat.


In [3]:
# Bangun Model Faster R-CNN
import torch
import torch_directml

from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn,
    FasterRCNN_ResNet50_FPN_Weights
)

from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# Device DirectML
device = torch_directml.device()

# Jumlah kelas: 8 kelas APD + 1 background
num_classes = 9

# Membuat arsitektur Faster R-CNN
model = fasterrcnn_resnet50_fpn(
    weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT,
    min_size=640,
    max_size=640
)

# Mengganti predictor agar sesuai dengan dataset CHVG
in_features = model.roi_heads.box_predictor.cls_score.in_features

model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
    num_classes
)

# Memindahkan model ke DirectML
model.to(device)

print("=== MODEL BERHASIL DIBANGUN ===")
print("Arsitektur       : Faster R-CNN ResNet50 FPN")
print("Jumlah kelas     :", num_classes)
print("Ukuran input     : 640 x 640")
print("Device           :", next(model.parameters()).device)
print("Predictor output :", model.roi_heads.box_predictor.cls_score.out_features)

=== MODEL BERHASIL DIBANGUN ===
Arsitektur       : Faster R-CNN ResNet50 FPN
Jumlah kelas     : 9
Ukuran input     : 640 x 640
Device           : privateuseone:0
Predictor output : 9


In [4]:
# Pasang kembali 4 patch pada DirectML
import torch
import torch.nn.functional as F
import types

from types import MethodType
from collections import OrderedDict
import torchvision.models.detection.roi_heads as roi_heads_module


# ============================================================
# PATCH 1: RPN COMPUTE LOSS
# Menangani kondisi tidak ada positive anchors / sampled anchors
# ============================================================

def compute_loss_safe(self, objectness, pred_bbox_deltas,
                      labels, regression_targets):

    sampled_pos_inds, sampled_neg_inds = self.fg_bg_sampler(labels)

    sampled_pos_inds = torch.where(
        torch.cat(sampled_pos_inds, dim=0)
    )[0]

    sampled_neg_inds = torch.where(
        torch.cat(sampled_neg_inds, dim=0)
    )[0]

    sampled_inds = torch.cat(
        [sampled_pos_inds, sampled_neg_inds]
    )

    labels = torch.cat(labels, dim=0)
    regression_targets = torch.cat(regression_targets, dim=0)

    # Objectness loss
    if sampled_inds.numel() > 0:
        objectness_loss = F.binary_cross_entropy_with_logits(
            objectness[sampled_inds],
            labels[sampled_inds]
        )
    else:
        objectness_loss = objectness.sum() * 0.0

    # Bounding box regression loss
    if sampled_pos_inds.numel() > 0:
        box_loss = F.smooth_l1_loss(
            pred_bbox_deltas[sampled_pos_inds],
            regression_targets[sampled_pos_inds],
            beta=1 / 9,
            reduction="sum"
        ) / max(sampled_inds.numel(), 1)
    else:
        box_loss = pred_bbox_deltas.sum() * 0.0

    return objectness_loss, box_loss


model.rpn.compute_loss = MethodType(
    compute_loss_safe, model.rpn
)


# ============================================================
# PATCH 2: MODEL FORWARD
# Melewati pemeriksaan degenerate boxes jika target kosong
# ============================================================

def forward_safe(self, images, targets=None):

    if self.training and targets is None:
        raise ValueError(
            "Targets harus diberikan ketika model training."
        )

    if targets is not None:
        for target in targets:
            boxes = target["boxes"]

            if boxes.numel() > 0:
                degenerate_boxes = boxes[:, 2:] <= boxes[:, :2]

                if degenerate_boxes.any():
                    raise ValueError(
                        "Bounding box tidak valid ditemukan."
                    )

    original_image_sizes = []

    for image in images:
        val = image.shape[-2:]
        original_image_sizes.append((val[0], val[1]))

    images, targets = self.transform(images, targets)

    features = self.backbone(images.tensors)

    if isinstance(features, torch.Tensor):
        features = OrderedDict([("0", features)])

    proposals, proposal_losses = self.rpn(
        images, features, targets
    )

    detections, detector_losses = self.roi_heads(
        features,
        proposals,
        images.image_sizes,
        targets
    )

    detections = self.transform.postprocess(
        detections,
        images.image_sizes,
        original_image_sizes
    )

    losses = {}
    losses.update(detector_losses)
    losses.update(proposal_losses)

    if self.training:
        return losses

    return detections


model.forward = MethodType(forward_safe, model)


# ============================================================
# PATCH 3: ROI HEADS ADD GT PROPOSALS
# Menangani gambar tanpa ground-truth boxes
# ============================================================

def add_gt_proposals_safe(self, proposals, gt_boxes):

    if all(gt_box.numel() == 0 for gt_box in gt_boxes):
        return proposals

    new_proposals = []

    for proposal, gt_box in zip(proposals, gt_boxes):

        if gt_box.numel() == 0:
            new_proposals.append(proposal)
        else:
            new_proposals.append(
                torch.cat((proposal, gt_box), dim=0)
            )

    return new_proposals


model.roi_heads.add_gt_proposals = MethodType(
    add_gt_proposals_safe,
    model.roi_heads
)


# ============================================================
# PATCH 4: FAST R-CNN LOSS
# Menangani kondisi tidak ada positive proposals
# ============================================================

def fastrcnn_loss_safe(class_logits, box_regression,
                       labels, regression_targets):

    labels = torch.cat(labels, dim=0)
    regression_targets = torch.cat(
        regression_targets, dim=0
    )

    classification_loss = F.cross_entropy(
        class_logits, labels
    )

    sampled_pos_inds_subset = torch.where(labels > 0)[0]
    labels_pos = labels[sampled_pos_inds_subset]

    N, num_classes = box_regression.shape

    box_regression = box_regression.reshape(
        N, box_regression.size(-1) // 4, 4
    )

    if sampled_pos_inds_subset.numel() > 0:

        box_loss = F.smooth_l1_loss(
            box_regression[
                sampled_pos_inds_subset, labels_pos
            ],
            regression_targets[sampled_pos_inds_subset],
            beta=1 / 9,
            reduction="sum"
        ) / labels.numel()

    else:
        box_loss = box_regression.sum() * 0.0

    return classification_loss, box_loss


roi_heads_module.fastrcnn_loss = fastrcnn_loss_safe


# ============================================================
# VERIFIKASI PATCH
# ============================================================

print("=== PEMASANGAN PATCH DIRECTML ===")
print("RPN          :", model.rpn.compute_loss.__func__.__name__)
print("Model Forward:", model.forward.__func__.__name__)
print("ROI Heads    :", model.roi_heads.add_gt_proposals.__func__.__name__)
print("Fast R-CNN   :", roi_heads_module.fastrcnn_loss.__name__)

=== PEMASANGAN PATCH DIRECTML ===
RPN          : compute_loss_safe
Model Forward: forward_safe
ROI Heads    : add_gt_proposals_safe
Fast R-CNN   : fastrcnn_loss_safe


In [5]:
# Muat checkpoint epoch 5 dan pulihkan optimizer
import torch
import os

# Lokasi checkpoint
checkpoint_path = (
    r"C:\Faster-RCNN-PPE\experiments\fasterrcnn_20_epochs"
    r"\fasterrcnn_epoch_5.pth"
)

# Memuat checkpoint ke CPU terlebih dahulu
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

# Memulihkan bobot model
model.load_state_dict(checkpoint["model_state_dict"])

# Memulihkan optimizer SGD
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.001,
    momentum=0.9,
    weight_decay=0.0005
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

# Memastikan learning rate sesuai checkpoint
for param_group in optimizer.param_groups:
    param_group["lr"] = checkpoint["learning_rate"]

# Memastikan model berada di DirectML
model.to(device)

# Informasi checkpoint
start_epoch = checkpoint["epoch"]

print("=== CHECKPOINT BERHASIL DIMUAT ===")
print("Epoch terakhir       :", start_epoch)
print("Average epoch loss   :", checkpoint["average_epoch_loss"])
print("Total epoch loss     :", checkpoint["total_epoch_loss"])
print("Jumlah kelas         :", checkpoint["num_classes"])
print("Learning rate        :", optimizer.param_groups[0]["lr"])
print("Optimizer            :", type(optimizer).__name__)
print("Device model         :", next(model.parameters()).device)

=== CHECKPOINT BERHASIL DIMUAT ===
Epoch terakhir       : 5
Average epoch loss   : 0.2913662218234186
Total epoch loss     : 346.43443774804473
Jumlah kelas         : 9
Learning rate        : 0.001
Optimizer            : SGD
Device model         : privateuseone:0


In [6]:
# Pemeriksaan sebelum lanjut ke epoch 10
import os
import torch

print("=== STATUS MODEL ===")
print("Device:", next(model.parameters()).device)
print("Mode training:", model.training)
print("Jumlah kelas:", model.roi_heads.box_predictor.cls_score.out_features)

print("\n=== STATUS OPTIMIZER ===")
print("Optimizer:", type(optimizer).__name__)
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Momentum:", optimizer.param_groups[0]["momentum"])

print("\n=== STATUS DATASET ===")
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))
print("Train batches:", len(train_loader))

print("\n=== STATUS PATCH DIRECTML ===")
print("RPN:", model.rpn.compute_loss.__func__.__name__)
print("Forward:", model.forward.__func__.__name__)
print("ROI Heads:", model.roi_heads.add_gt_proposals.__func__.__name__)
print("Fast R-CNN:", roi_heads_module.fastrcnn_loss.__name__)

print("\n=== STATUS TRAINING ===")
print("Epoch terakhir:", start_epoch)
print("Epoch berikutnya:", start_epoch + 1)
print("Checkpoint tersedia:", os.path.exists(checkpoint_path))

=== STATUS MODEL ===
Device: privateuseone:0
Mode training: True
Jumlah kelas: 9

=== STATUS OPTIMIZER ===
Optimizer: SGD
Learning rate: 0.001
Momentum: 0.9

=== STATUS DATASET ===
Train: 1189
Validation: 170
Test: 340
Train batches: 1189

=== STATUS PATCH DIRECTML ===
RPN: compute_loss_safe
Forward: forward_safe
ROI Heads: add_gt_proposals_safe
Fast R-CNN: fastrcnn_loss_safe

=== STATUS TRAINING ===
Epoch terakhir: 5
Epoch berikutnya: 6
Checkpoint tersedia: True


In [7]:
# Training epoch 6–10
import os
import time
import torch

def move_to_cpu(data):
    if isinstance(data, torch.Tensor):
        return data.detach().cpu()
    elif isinstance(data, dict):
        return {k: move_to_cpu(v) for k, v in data.items()}
    elif isinstance(data, list):
        return [move_to_cpu(v) for v in data]
    elif isinstance(data, tuple):
        return tuple(move_to_cpu(v) for v in data)
    return data


training_history = []

for epoch in range(start_epoch + 1, 11):

    model.train()

    total_loss = 0.0
    start_time = time.time()

    print(f"\n{'='*55}")
    print(f"MEMULAI TRAINING EPOCH {epoch}/20")
    print(f"{'='*55}")

    for batch_idx, (images, targets) in enumerate(train_loader):

        images = [img.to(device) for img in images]

        targets = [
            {k: v.to(device) for k, v in target.items()}
            for target in targets
        ]

        optimizer.zero_grad()

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        losses.backward()
        optimizer.step()

        batch_loss = losses.item()
        total_loss += batch_loss

        if (batch_idx + 1) % 100 == 0 or (batch_idx + 1) == len(train_loader):
            print(
                f"Epoch {epoch} | "
                f"Batch {batch_idx + 1}/{len(train_loader)} | "
                f"Loss: {batch_loss:.6f} | "
                f"Rata-rata loss: {total_loss / (batch_idx + 1):.6f}"
            )

    average_loss = total_loss / len(train_loader)
    elapsed_time = (time.time() - start_time) / 60

    training_history.append({
        "epoch": epoch,
        "total_loss": total_loss,
        "average_loss": average_loss,
        "time_minutes": elapsed_time
    })

    print(f"\nHASIL EPOCH {epoch}")
    print(f"Total loss       : {total_loss:.6f}")
    print(f"Average loss     : {average_loss:.6f}")
    print(f"Waktu training   : {elapsed_time:.2f} menit")

    # Menyimpan checkpoint terbaru
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": move_to_cpu(model.state_dict()),
        "optimizer_state_dict": move_to_cpu(optimizer.state_dict()),
        "average_epoch_loss": average_loss,
        "total_epoch_loss": total_loss,
        "num_classes": 9,
        "learning_rate": optimizer.param_groups[0]["lr"]
    }

    latest_path = os.path.join(
        output_dir, "fasterrcnn_latest.pth"
    )

    temp_path = latest_path + ".tmp"

    torch.save(checkpoint, temp_path)
    os.replace(temp_path, latest_path)

    print("Checkpoint terbaru tersimpan:", latest_path)

    # Checkpoint khusus epoch 10
    if epoch == 10:

        milestone_path = os.path.join(
            output_dir, "fasterrcnn_epoch_10.pth"
        )

        torch.save(checkpoint, milestone_path)

        print(
            "Checkpoint milestone tersimpan:",
            milestone_path
        )

    print(f"Epoch {epoch} selesai.")

    start_epoch = epoch

print("\nTRAINING EPOCH 6–10 SELESAI.")


MEMULAI TRAINING EPOCH 6/20


ValueError: Target size (torch.Size([256])) must be the same as input size (torch.Size([256, 1]))

In [8]:
# Perbaiki patch RPN
import torch
import torch.nn.functional as F
from types import MethodType

def compute_loss_safe(self, objectness, pred_bbox_deltas,
                      labels, regression_targets):

    sampled_pos_inds, sampled_neg_inds = self.fg_bg_sampler(labels)

    sampled_pos_inds = torch.where(
        torch.cat(sampled_pos_inds, dim=0)
    )[0]

    sampled_neg_inds = torch.where(
        torch.cat(sampled_neg_inds, dim=0)
    )[0]

    sampled_inds = torch.cat(
        [sampled_pos_inds, sampled_neg_inds]
    )

    # Samakan bentuk tensor
    objectness = objectness.reshape(-1)
    labels = torch.cat(labels, dim=0).reshape(-1)
    pred_bbox_deltas = pred_bbox_deltas.reshape(-1, 4)
    regression_targets = torch.cat(
        regression_targets, dim=0
    ).reshape(-1, 4)

    # Objectness loss
    if sampled_inds.numel() > 0:
        objectness_loss = F.binary_cross_entropy_with_logits(
            objectness[sampled_inds],
            labels[sampled_inds].to(objectness.dtype)
        )
    else:
        objectness_loss = objectness.sum() * 0.0

    # Bounding box regression loss
    if sampled_pos_inds.numel() > 0:
        box_loss = F.smooth_l1_loss(
            pred_bbox_deltas[sampled_pos_inds],
            regression_targets[sampled_pos_inds],
            beta=1 / 9,
            reduction="sum"
        ) / max(sampled_inds.numel(), 1)
    else:
        box_loss = pred_bbox_deltas.sum() * 0.0

    return objectness_loss, box_loss


# Pasang ulang patch RPN
model.rpn.compute_loss = MethodType(
    compute_loss_safe,
    model.rpn
)

print("Patch RPN berhasil diperbarui:")
print(model.rpn.compute_loss.__func__.__name__)

Patch RPN berhasil diperbarui:
compute_loss_safe


In [9]:
# Uji forward pass sebelum training 6-10 epoch
model.train()

# Ambil satu batch untuk pemeriksaan
images, targets = next(iter(train_loader))

images = [img.to(device) for img in images]

targets = [
    {k: v.to(device) for k, v in target.items()}
    for target in targets
]

# Forward pass tanpa backward dan optimizer.step()
with torch.no_grad():
    loss_dict = model(images, targets)

print("=== HASIL UJI FORWARD PASS ===")

for name, value in loss_dict.items():
    print(f"{name}: {value.item():.6f}")

total_loss = sum(loss_dict.values())

print("Total loss:", total_loss.item())
print("Status: Forward pass berhasil.")

C:\Users\ANGELINA\miniconda3\envs\pytorch-directml\lib\site-packages\torch\nn\functional.py:3244: UserWarning: The operator 'aten::log_sigmoid_forward' is not currently supported on the DML backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at C:\__w\1\s\pytorch-directml-plugin\torch_directml\csrc\dml\dml_cpu_fallback.cpp:17.)
  return torch.binary_cross_entropy_with_logits(input, target, weight, pos_weight, reduction_enum)


=== HASIL UJI FORWARD PASS ===
loss_classifier: 0.050524
loss_box_reg: 0.111585
loss_objectness: 0.000145
loss_rpn_box_reg: 0.005958
Total loss: 0.16821210086345673
Status: Forward pass berhasil.


In [10]:
output_dir = r"C:\Faster-RCNN-PPE\experiments\fasterrcnn_20_epochs"

start_epoch = 5

print("Epoch awal:", start_epoch)
print("Epoch berikutnya:", start_epoch + 1)
print("Folder output:", output_dir)
print("Patch RPN:", model.rpn.compute_loss.__func__.__name__)
print("Optimizer:", type(optimizer).__name__)

Epoch awal: 5
Epoch berikutnya: 6
Folder output: C:\Faster-RCNN-PPE\experiments\fasterrcnn_20_epochs
Patch RPN: compute_loss_safe
Optimizer: SGD


In [11]:
# Training epoch 6-10
import os
import time
import torch

def move_to_cpu(data):
    if isinstance(data, torch.Tensor):
        return data.detach().cpu()
    elif isinstance(data, dict):
        return {k: move_to_cpu(v) for k, v in data.items()}
    elif isinstance(data, list):
        return [move_to_cpu(v) for v in data]
    elif isinstance(data, tuple):
        return tuple(move_to_cpu(v) for v in data)
    return data


training_history = []

for epoch in range(start_epoch + 1, 11):

    model.train()
    total_loss = 0.0
    start_time = time.time()

    print(f"\n{'='*55}")
    print(f"MEMULAI TRAINING EPOCH {epoch}/20")
    print(f"{'='*55}")

    for batch_idx, (images, targets) in enumerate(train_loader):

        images = [img.to(device) for img in images]

        targets = [
            {k: v.to(device) for k, v in target.items()}
            for target in targets
        ]

        optimizer.zero_grad()

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        losses.backward()
        optimizer.step()

        batch_loss = losses.item()
        total_loss += batch_loss

        if (batch_idx + 1) % 100 == 0 or (batch_idx + 1) == len(train_loader):
            print(
                f"Epoch {epoch} | "
                f"Batch {batch_idx + 1}/{len(train_loader)} | "
                f"Loss: {batch_loss:.6f} | "
                f"Rata-rata loss: {total_loss / (batch_idx + 1):.6f}"
            )

    average_loss = total_loss / len(train_loader)
    elapsed_time = (time.time() - start_time) / 60

    training_history.append({
        "epoch": epoch,
        "total_loss": total_loss,
        "average_loss": average_loss,
        "time_minutes": elapsed_time
    })

    print(f"\nHASIL EPOCH {epoch}")
    print(f"Total loss       : {total_loss:.6f}")
    print(f"Average loss     : {average_loss:.6f}")
    print(f"Waktu training   : {elapsed_time:.2f} menit")

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": move_to_cpu(model.state_dict()),
        "optimizer_state_dict": move_to_cpu(optimizer.state_dict()),
        "average_epoch_loss": average_loss,
        "total_epoch_loss": total_loss,
        "num_classes": 9,
        "learning_rate": optimizer.param_groups[0]["lr"]
    }

    latest_path = os.path.join(
        output_dir, "fasterrcnn_latest.pth"
    )

    temp_path = latest_path + ".tmp"

    torch.save(checkpoint, temp_path)
    os.replace(temp_path, latest_path)

    print("Checkpoint terbaru tersimpan:", latest_path)

    if epoch == 10:
        milestone_path = os.path.join(
            output_dir, "fasterrcnn_epoch_10.pth"
        )

        torch.save(checkpoint, milestone_path)
        print("Checkpoint milestone tersimpan:", milestone_path)

    start_epoch = epoch
    print(f"Epoch {epoch} selesai.")

print("\nTRAINING EPOCH 6–10 SELESAI.")


MEMULAI TRAINING EPOCH 6/20
Epoch 6 | Batch 100/1189 | Loss: 0.566470 | Rata-rata loss: 0.269724
Epoch 6 | Batch 200/1189 | Loss: 0.431881 | Rata-rata loss: 0.264980
Epoch 6 | Batch 300/1189 | Loss: 0.153800 | Rata-rata loss: 0.271358
Epoch 6 | Batch 400/1189 | Loss: 0.080017 | Rata-rata loss: 0.271939
Epoch 6 | Batch 500/1189 | Loss: 0.092589 | Rata-rata loss: 0.273125
Epoch 6 | Batch 600/1189 | Loss: 0.228112 | Rata-rata loss: 0.269409
Epoch 6 | Batch 700/1189 | Loss: 0.259670 | Rata-rata loss: 0.268603
Epoch 6 | Batch 800/1189 | Loss: 0.195165 | Rata-rata loss: 0.268060
Epoch 6 | Batch 900/1189 | Loss: 0.439716 | Rata-rata loss: 0.265000
Epoch 6 | Batch 1000/1189 | Loss: 0.099815 | Rata-rata loss: 0.266627
Epoch 6 | Batch 1100/1189 | Loss: 0.549724 | Rata-rata loss: 0.268964
Epoch 6 | Batch 1189/1189 | Loss: 0.314711 | Rata-rata loss: 0.267927

HASIL EPOCH 6
Total loss       : 318.564654
Average loss     : 0.267927
Waktu training   : 41.59 menit
Checkpoint terbaru tersimpan: C:\Fas